In [ ]:
#!/usr/bin/env python3
"""HK Equities Daily — one run creates a formatted, unsent Outlook draft.

Designed for a SINGLE Jupyter code cell or `python HK_Equities_Bloomberg.py`.
Works with Python 3.8+ standard libraries to gather public data. On Windows it
opens an editable draft in Classic Outlook via pywin32 or PowerShell COM.
No message is sent. If Outlook automation is unsupported, an HTML preview,
.eml file, and other audit files are created in the output folder.

Bloomberg setup: run on your logged-in Bloomberg desktop with Desktop API access.
Install blpapi into the same Python environment as Jupyter, following:
https://professional.bloomberg.com/support/api-library/
python -m pip install --index-url=https://blpapi.bloomberg.com/repository/releases/python/simple/ blpapi

The final section includes unlock / free-float market cap (%) and unlock / 20-day
average turnover (x). Monetary inputs are normalized to absolute HKD. Bloomberg
errors appear as N/A with review reasons; set BLOOMBERG_REQUIRED to True to
withhold the draft if any ratio is unavailable. Nothing is sent automatically.
"""

from __future__ import annotations

import os

# ===================== EDIT ONLY THESE SETTINGS =====================
EMAIL_SUBJECT = "IPO, PHIP & Lock-up Update | {date}"
EMAIL_TO = os.environ.get("HK_EQUITIES_EMAIL_TO", "oscar.chan1@nomura.com")
EMAIL_CC = os.environ.get("HK_EQUITIES_EMAIL_CC", "kenny.tam1@nomura.com")
OUTPUT_FOLDER = "HK_Equities_Drafts"   # Created beneath the Jupyter working directory
OPEN_OUTLOOK = True                    # Show a real editable Outlook draft when possible
SHOW_NOTEBOOK_PREVIEW_IF_NO_OUTLOOK = True
INCLUDE_GEM_PHIPS = True
INCLUDE_EXPIRED_THIS_MONTH = False
SHORT_HISTORY_LOOKBACK_DAYS = 45
# In a corporate PowerShell Constrained Language environment, the old
# PowerShell web-download fallback fails. Stay on the confirmed Python proxy.
NETWORK_MODE = "urllib"       # Recommended. "auto" / "windows_proxy" are legacy options.
NETWORK_TIMEOUT_SECONDS = 18 # Per-socket inactivity limit; prevents a stalled source holding everything up
ETNET_TIMEOUT_SECONDS = 22    # Slightly more leeway for the larger etnet pages
NETWORK_RETRIES = 2           # Other public sources: retry recoverable download errors.
ETNET_RETRIES = 2             # Avoid multi-minute loops on repeated proxy 503s.
NETWORK_RETRY_PAUSE_SECONDS = 1
SHOW_DOWNLOAD_TIMINGS = True   # Identify a slow/blocked source directly in Jupyter
ETNET_CACHE_ENABLED = True     # Save fully verified etnet data immediately when each month succeeds
ETNET_CACHE_MAX_MINUTES = 120  # ONLY reuse within this many minutes, and ONLY on the same HKT day
USE_GZIP_COMPRESSION = True    # Request smaller HTTP transfers; verify decompression and full response
WINDOWS_PROXY_TIMEOUT_SECONDS = 35  # Legacy fallback, not used in "urllib" mode
# Optional: if Edge opens the etnet page but Python still times out, save the
# CURRENT page as HTML in this directory using a filename like etnet_202610.html.
# Only a recent file from a matching month is accepted and fully validated.
ALLOW_MANUAL_ETNET_HTML = True
MANUAL_ETNET_FOLDER = "HK_Equities_Drafts/manual_etnet"
MANUAL_ETNET_MAX_AGE_HOURS = 24
# Keep the original working corporate proxy unless explicitly overridden.
# A missing or blank HK_EQUITIES_HTTPS_PROXY keeps the original default.
# Use http:// here: the HTTP proxy CONNECT-tunnels HTTPS destinations.
# Never store company passwords or tokens in this notebook.
# If your approved proxy changes, update this single line.
HTTPS_PROXY_URL = os.environ.get("HK_EQUITIES_HTTPS_PROXY", "").strip() or ("ht"+"tp://185.46.212.90:80")

# Bloomberg connects to the local Desktop API, not the public-web HTTP proxy.
BLOOMBERG_ENABLED = True
BLOOMBERG_REQUIRED = False          # True: withhold draft if any ratio is unavailable
BLOOMBERG_HOST = "localhost"
BLOOMBERG_PORT = 8194
BLOOMBERG_CONNECT_TIMEOUT_MS = 5000
BLOOMBERG_REQUEST_TIMEOUT_SECONDS = 30
BLOOMBERG_BATCH_SIZE = 100           # Unique securities per request
# Blank = Bloomberg's latest 20D interval; optional historical end date YYYYMMDD.
# This affects turnover ONLY. Market cap/free-float remain current snapshots;
# etnet unlock values retain the source quote timestamp shown in the report.
BLOOMBERG_TURNOVER_END_DATE = ""
# ====================================================================

from contextlib import contextmanager
from concurrent.futures import ThreadPoolExecutor
import base64
import subprocess
from dataclasses import asdict, dataclass
from datetime import date, datetime, timedelta, timezone
from decimal import Decimal, InvalidOperation
from email.message import EmailMessage
from email.policy import SMTP
from email.utils import format_datetime, make_msgid
from html.parser import HTMLParser
import http.client
import zlib
import hashlib
import html
import json
import logging
import tempfile
from pathlib import Path
import re
import sqlite3
import ssl
import sys
import time
import unicodedata
import urllib.error
import urllib.request
from urllib.parse import urljoin, urlparse, parse_qs
import webbrowser

HK = timezone(timedelta(hours=8), "HKT")
ROOT = Path.cwd()  # Works inside a Jupyter notebook or as a standalone .py file
DEFAULT_DAILY_TIME = "18:30"
MAIN_BOARD_URL = "https://www2.hkexnews.hk/New-Listings/New-Listing-Information/Main-Board?sc_lang=zh-HK"
PHIP_PAGE_URL = "https://www1.hkexnews.hk/app/appindex.html?lang=zh"
PHIP_FEED_BASE = "https://www1.hkexnews.hk/ncms/json/eds/"
PHIP_DOCUMENT_BASE = "https://www1.hkexnews.hk/app/"
SHORT_URL = "https://www.hkex.com.hk/eng/market/sec_tradinfo/ds{day:%Y%m%d}.htm"
LOCKUP_URL = "https://www.etnet.com.hk/www/tc/stocks/ipo-free-float.php?month={month}"
HOLIDAYS_URL = "https://www.1823.gov.hk/common/ical/en.json"
LOG = logging.getLogger("hk_equities_daily")


class Element:
    """Small HTML tree for the specific public source tables and cards."""
    def __init__(self, tag="document", attrs=(), parent=None):
        self.tag, self.attrs, self.parent = tag, dict(attrs), parent
        self.children = []

    def __getitem__(self, key):
        return self.attrs[key]

    def get(self, key, default=None):
        return self.attrs.get(key, default)

    def descendants(self):
        for child in self.children:
            if isinstance(child, Element):
                yield child
                yield from child.descendants()

    def find_all(self, tags, recursive=True):
        tags = {tags} if isinstance(tags, str) else set(tags)
        nodes = self.descendants() if recursive else self.children
        return [node for node in nodes if isinstance(node, Element) and node.tag in tags]

    def find(self, tag):
        return next((node for node in self.descendants() if node.tag == tag), None)

    def get_text(self, separator="", strip=False):
        def walk(node):
            for child in node.children:
                if isinstance(child, str):
                    if not strip or child.strip(): yield child.strip() if strip else child
                elif child.tag not in {"script", "style"}:
                    yield from walk(child)
        return separator.join(walk(self))

    @staticmethod
    def matches(node, selector):
        attribute = re.search(r'\[([\w-]+)(?:([*]?=)["\x27]?([^"\x27\]]+)["\x27]?)?\]', selector)
        if attribute:
            key, operator, value = attribute.groups()
            if key not in node.attrs: return False
            actual = node.attrs[key] or ""
            if operator == "=" and actual != value: return False
            if operator == "*=" and value not in actual: return False
            selector = selector[:attribute.start()] + selector[attribute.end():]
        pieces = selector.split(".")
        if pieces[0] and node.tag != pieces[0]: return False
        return all(piece in (node.attrs.get("class") or "").split() for piece in pieces[1:])

    def select(self, selector):
        matches = [self]
        for token in selector.split():
            found, seen = [], set()
            for parent in matches:
                for node in parent.descendants():
                    if id(node) not in seen and self.matches(node, token):
                        seen.add(id(node)); found.append(node)
            matches = found
        return matches

    def select_one(self, selector):
        return next(iter(self.select(selector)), None)


class DocumentParser(HTMLParser):
    VOID = {"area", "base", "br", "col", "embed", "hr", "img", "input", "link", "meta", "param", "source", "track", "wbr"}

    def __init__(self):
        super().__init__(convert_charrefs=True)
        self.root = Element()
        self.stack = [self.root]

    def handle_starttag(self, tag, attrs):
        node = Element(tag, attrs, self.stack[-1])
        self.stack[-1].children.append(node)
        if tag not in self.VOID: self.stack.append(node)

    def handle_startendtag(self, tag, attrs):
        self.handle_starttag(tag, attrs)
        if tag not in self.VOID: self.handle_endtag(tag)

    def handle_endtag(self, tag):
        for index in range(len(self.stack)-1, 0, -1):
            if self.stack[index].tag == tag:
                del self.stack[index:]
                return

    def handle_data(self, data):
        self.stack[-1].children.append(data)


def parse_html(document, *_):
    parser = DocumentParser()
    parser.feed(document)
    parser.close()
    return parser.root


def tls_context():
    context = ssl.create_default_context()
    # Python.org macOS installs sometimes lack an OpenSSL default CA file.
    # Add the OS certificate bundle without disabling certificate validation.
    if sys.platform == "darwin" and Path("/etc/ssl/cert.pem").is_file():
        context.load_verify_locations(cafile="/etc/ssl/cert.pem")
    return context


def _decode_source(url, raw, encoding=None):
    """Keep identical source decoding for urllib and Windows proxy downloads."""
    if raw.startswith(b"\xef\xbb\xbf") or urlparse(url).path.endswith('.json'):
        return raw.decode('utf-8-sig')
    declared = re.search(rb'charset\s*=\s*["\x27]?([\w-]+)', raw[:8192], re.I)
    encoding = declared[1].decode('ascii') if declared else (encoding or 'utf-8')
    return raw.decode(encoding, errors='strict')


def _read_complete_response(response, url, max_bytes=20_000_000):
    """Read the full HTTP response, never trusting or returning truncated data.

    A company proxy sometimes closes HTTP/1.1 chunked responses before the
    terminating chunk is received. `http.client.IncompleteRead` must cause a
    fresh request, not be mistaken for a valid partial webpage. Bounded small
    reads also avoid allocating a 20 MB read buffer on every request.
    """
    chunks = []
    total = 0
    while True:
        part = response.read(64 * 1024)
        if not part:
            break
        total += len(part)
        if total > max_bytes:
            raise SourceError('Source exceeds the allowed download size: ' + url)
        chunks.append(part)
    # For non-chunked responses a premature EOF may not always raise.
    # Only check numeric Content-Length when HTTP transfers are not chunked.
    transfer_encoding = (response.headers.get('Transfer-Encoding') or '').lower()
    length = response.headers.get('Content-Length')
    if 'chunked' not in transfer_encoding and length and length.isdecimal():
        if total != int(length):
            raise http.client.IncompleteRead(b'', int(length) - total)
    return b''.join(chunks)


def _decode_http_encoding(raw, encoding, url, limit=20_000_000):
    """Handle HTTP gzip/deflate with a strict decompressed-size and stream-integrity check."""
    encoding = (encoding or '').strip().lower()
    if not encoding or encoding == 'identity':
        return raw
    if encoding not in ('gzip', 'deflate'):
        raise SourceError('Unsupported Content-Encoding ' + repr(encoding) + ': ' + url)
    decoder = zlib.decompressobj(16 + zlib.MAX_WBITS if encoding == 'gzip' else zlib.MAX_WBITS)
    try:
        data = decoder.decompress(raw, limit + 1)
        if len(data) > limit or decoder.unconsumed_tail:
            raise SourceError('Uncompressed webpage exceeds safety limit: ' + url)
        data += decoder.flush(limit + 1 - len(data))
        if not decoder.eof:
            raise http.client.IncompleteRead(data)
        if len(data) > limit or decoder.unused_data:
            raise SourceError('Unexpected extra bytes in compressed response: ' + url)
        return data
    except zlib.error as exc:
        raise http.client.IncompleteRead(b'') from exc


def _etnet_month(url):
    parsed = urlparse(url)
    if parsed.hostname != 'www.etnet.com.hk' or parsed.path != '/www/tc/stocks/ipo-free-float.php':
        return None
    month = parse_qs(parsed.query).get('month', [''])[0]
    return month if re.fullmatch(r'\d{6}', month) else None


def _verified_etnet_cache_paths(month):
    directory = ROOT / OUTPUT_FOLDER / '.history' / 'verified_etnet_cache'
    base = 'etnet_' + month + '_'
    return directory / (base + 'verified.html'), directory / (base + 'verified.json')


def _download_via_windows_proxy(url, timeout):
    """Use Windows proxy/PAC settings and current login; never disable TLS checks.

    This helps managed desktops where Edge/Outlook can access the web but
    Python urllib cannot evaluate the company's automatic PAC configuration.
    Windows PowerShell 5.1 ships with Windows and needs no Python packages.
    """
    if os.name != 'nt':
        raise SourceNetworkError('Windows proxy fallback is available only on Windows.')
    with tempfile.TemporaryDirectory(prefix='hk_eq_net_') as temp:
        download = Path(temp) / 'response.bin'
        command = '\n'.join([
            "$ErrorActionPreference = 'Stop'",
            '[System.Net.ServicePointManager]::SecurityProtocol = [System.Net.SecurityProtocolType]::Tls12',
            '$url = ' + _ps_quote(url),
            '$target = ' + _ps_quote(str(download)),
            ('$proxy = [System.Net.WebProxy]::new(' + _ps_quote(HTTPS_PROXY_URL.strip()) + ', $true)'
             if HTTPS_PROXY_URL.strip() else
             '$proxy = [System.Net.WebRequest]::GetSystemWebProxy()'),
            'if ($null -ne $proxy) { $proxy.Credentials = [System.Net.CredentialCache]::DefaultNetworkCredentials }',
            '$req = [System.Net.HttpWebRequest]::Create($url)',
            '$req.Method = "GET"',
            '$req.UserAgent = "Mozilla/5.0 (compatible; HK-Equities-Daily/3.0)"',
            '$req.Accept = "text/html,application/json,*/*"',
            '$req.Headers.Add("Accept-Language", "zh-HK,zh;q=0.9,en;q=0.8")',
            '$req.Timeout = ' + str(int(timeout * 1000)),
            '$req.ReadWriteTimeout = ' + str(int(timeout * 1000)),
            '$req.Proxy = $proxy',
            'try {',
            '  $resp = $req.GetResponse()',
            '  try {',
            '    $inputStream = $resp.GetResponseStream()',
            '    $outputStream = [System.IO.File]::Create($target)',
            '    try { $inputStream.CopyTo($outputStream) }',
            '    finally { $outputStream.Dispose(); $inputStream.Dispose() }',
            '  } finally { $resp.Dispose() }',
            '} catch {',
            '  [Console]::Error.WriteLine($_.Exception.Message)',
            '  exit 1',
            '}',
        ])
        encoded = base64.b64encode(command.encode('utf-16le')).decode('ascii')
        try:
            result = subprocess.run(
                ['powershell.exe', '-NoProfile', '-NonInteractive', '-EncodedCommand', encoded],
                capture_output=True, text=True, timeout=timeout+10, check=False,
            )
        except (OSError, subprocess.TimeoutExpired) as exc:
            raise SourceNetworkError('Windows PowerShell web download unavailable: ' + str(exc)) from exc
        if result.returncode != 0:
            reason = (result.stderr or result.stdout or 'unknown download error').strip()[:600]
            if re.search(r'\(404\)|HTTP\s*404', reason, re.I):
                raise PageMissing(f'Not published (HTTP 404): {url}')
            if re.search(r'\(410\)|HTTP\s*410', reason, re.I):
                raise PageMissing(f'Not published (HTTP 410): {url}')
            raise SourceNetworkError('Windows proxy fetch failed for ' + url + ': ' + reason)
        if not download.is_file():
            raise SourceNetworkError('Windows proxy fetch returned no file for ' + url)
        return download.read_bytes()


class Client:
    def __init__(self):
        self.context = tls_context()
        self.cache = {}
        self.manual_sources = {}
        self.cached_sources = {}
        self.windows_proxy_worked = NETWORK_MODE == 'windows_proxy'
        if NETWORK_MODE not in ('auto', 'urllib', 'windows_proxy'):
            raise ValueError('NETWORK_MODE must be auto, urllib, or windows_proxy.')
        if not isinstance(NETWORK_RETRIES, int) or NETWORK_RETRIES < 1:
            raise ValueError('NETWORK_RETRIES must be a positive integer.')
        if not isinstance(ETNET_RETRIES, int) or ETNET_RETRIES < 1:
            raise ValueError('ETNET_RETRIES must be a positive integer.')
        # The user's successful proxy test used urllib.install_opener().
        # This Client builds its own opener; configure that opener explicitly,
        # otherwise it silently ignores the working global proxy!
        if HTTPS_PROXY_URL.strip():
            parsed_proxy = urlparse(HTTPS_PROXY_URL.strip())
            if (parsed_proxy.scheme not in ('http', 'https') or not parsed_proxy.hostname
                    or parsed_proxy.username or parsed_proxy.password):
                raise ValueError('HTTPS_PROXY_URL must be an http(s) proxy URL without credentials.')
            self.opener = urllib.request.build_opener(
                urllib.request.ProxyHandler({
                    'http': HTTPS_PROXY_URL.strip(),
                    'https': HTTPS_PROXY_URL.strip(),
                }),
                urllib.request.HTTPSHandler(context=self.context),
            )
        else:
            self.opener = urllib.request.build_opener(
                urllib.request.ProxyHandler(),
                urllib.request.HTTPSHandler(context=self.context),
            )

    def _recent_manual_etnet(self, url):
        """Use only a fresh, locally browser-saved etnet page for this exact month.

        It still passes through the normal strict visible/embedded-source parsing.
        The file must be obtained by the user through an approved browser session.
        """
        if not ALLOW_MANUAL_ETNET_HTML:
            return None
        parsed = urlparse(url)
        if parsed.scheme != 'https' or parsed.hostname != 'www.etnet.com.hk' or \
                parsed.path != '/www/tc/stocks/ipo-free-float.php':
            return None
        month = parse_qs(parsed.query).get('month', [''])[0]
        if not re.fullmatch(r'\d{6}', month):
            return None
        source = (ROOT / MANUAL_ETNET_FOLDER / ('etnet_' + month + '.html')).resolve()
        if not source.is_file():
            return None
        age_seconds = time.time() - source.stat().st_mtime
        max_age = MANUAL_ETNET_MAX_AGE_HOURS * 3600
        if not 0 <= age_seconds <= max_age:
            raise SourceNetworkError(
                'The manual etnet HTML file is older than ' + str(MANUAL_ETNET_MAX_AGE_HOURS)
                + ' hours or has an invalid modification time: ' + str(source)
                + '. Save a fresh page from Edge; do not use stale lock-up data.'
            )
        raw = source.read_bytes()
        if not raw or len(raw) > 20_000_000:
            raise SourceError('The manual etnet HTML file is empty or unexpectedly large: ' + str(source))
        try:
            page = _decode_source(url, raw)
        except (UnicodeError, LookupError) as exc:
            raise SourceError('Cannot decode manual etnet HTML file: ' + str(source)) from exc
        self.cache[url] = page
        self.manual_sources[month] = datetime.fromtimestamp(source.stat().st_mtime, HK).strftime('%d %b %Y %H:%M HKT')
        print('Using freshly browser-saved etnet HTML for ' + month
              + ' (age %.1f hours). Normal source validation still applies.' % (age_seconds/3600))
        return page

    def _recent_verified_etnet_cache(self, url):
        """Same-day, time-limited verified HTTP snapshot; reject mismatches and tampering."""
        month = _etnet_month(url)
        if not month or not ETNET_CACHE_ENABLED:
            return None
        html_path, metadata_path = _verified_etnet_cache_paths(month)
        if not html_path.is_file() or not metadata_path.is_file():
            return None
        try:
            metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
            if metadata.get('url') != url or metadata.get('report_day') != datetime.now(HK).date().isoformat():
                return None
            fetched = datetime.fromisoformat(metadata['fetched_at'])
            now = datetime.now(HK)
            age = (now - fetched).total_seconds()
            if fetched.tzinfo is None or not 0 <= age <= ETNET_CACHE_MAX_MINUTES * 60:
                return None
            raw = html_path.read_bytes()
            if len(raw) != metadata.get('bytes') or hashlib.sha256(raw).hexdigest() != metadata.get('sha256'):
                return None
            page = raw.decode('utf-8', errors='strict')
            # An HTTP cache hit is never proof by itself. Recheck the source table.
            parse_lockups(page, month)
            self.cache[url] = page
            stamp = fetched.astimezone(HK).strftime('%d %b %Y %H:%M HKT')
            self.cached_sources[month] = stamp
            print('  VERIFIED CACHE etnet ' + month + ' (' + stamp + ', age %.0f min)' % (age / 60), flush=True)
            return page
        except (OSError, ValueError, KeyError, TypeError, UnicodeError, SourceError) as exc:
            print('  NOTE etnet ' + month + ' cache unusable (' + str(exc)[:100] + '); fetching live...', flush=True)
            return None

    def _save_verified_etnet_cache(self, url, page):
        month = _etnet_month(url)
        if not month or not ETNET_CACHE_ENABLED:
            return
        # Save only sources that pass the same visible-vs-embedded check used for the draft.
        parse_lockups(page, month)
        html_path, metadata_path = _verified_etnet_cache_paths(month)
        try:
            html_path.parent.mkdir(parents=True, exist_ok=True)
            raw = page.encode('utf-8')
            stamp = datetime.now(HK)
            metadata = {'url': url, 'report_day': stamp.date().isoformat(),
                        'fetched_at': stamp.isoformat(), 'sha256': hashlib.sha256(raw).hexdigest(),
                        'bytes': len(raw), 'source': 'complete validated etnet HTML'}
            atomic_write(html_path, raw)
            atomic_write(metadata_path, json.dumps(metadata, indent=2).encode('utf-8'))
        except OSError as exc:
            # Failure to cache must never destroy already downloaded, validated data.
            print('  NOTE Could not save etnet cache: ' + str(exc)[:120], flush=True)

    def text(self, url):
        if url in self.cache:
            return self.cache[url]
        # Prefer a recent, validated browser-saved page immediately rather than
        # first waiting through two timeouts for a known troublesome etnet route.
        try:
            manual = self._recent_manual_etnet(url)
        except SourceNetworkError:
            # Stale browser HTML must never prevent a fresh network attempt.
            # It is not used as a substitute for current data.
            manual = None
        if manual is not None:
            return manual
        verified_cache = self._recent_verified_etnet_cache(url)
        if verified_cache is not None:
            return verified_cache

        short_source = urlparse(url)
        source_label = short_source.hostname + short_source.path.rsplit('/', 1)[-1]
        if short_source.hostname == 'www.etnet.com.hk':
            source_label = 'etnet ' + parse_qs(short_source.query).get('month', [''])[0]
        timeout = (ETNET_TIMEOUT_SECONDS if short_source.hostname == 'www.etnet.com.hk'
                   else NETWORK_TIMEOUT_SECONDS)
        retry_limit = (ETNET_RETRIES if short_source.hostname == 'www.etnet.com.hk'
                       else NETWORK_RETRIES)
        started_at = time.monotonic()
        if SHOW_DOWNLOAD_TIMINGS:
            print('  GET ' + source_label + ' ...', flush=True)
        req = urllib.request.Request(url, headers={
            'User-Agent': 'Mozilla/5.0 (compatible; HK-Equities-Daily/3.1)',
            'Accept-Language': 'zh-HK,zh;q=0.9,en;q=0.8',
            'Accept': 'text/html,application/json,*/*',
            'Accept-Encoding': 'gzip, deflate' if USE_GZIP_COMPRESSION else 'identity',
            'Connection': 'close',
            'Cache-Control': 'no-cache',
        })
        direct_failure = None
        use_windows = self.windows_proxy_worked and NETWORK_MODE != 'urllib'
        if not use_windows:
            for attempt in range(1, retry_limit + 1):
                try:
                    with self.opener.open(req, timeout=timeout) as response:
                        raw = _read_complete_response(response, url)
                        body = _decode_http_encoding(raw, response.headers.get('Content-Encoding'), url)
                        encoding = response.headers.get_content_charset()
                    data = _decode_source(url, body, encoding)
                    self._save_verified_etnet_cache(url, data)
                    self.cache[url] = data
                    if SHOW_DOWNLOAD_TIMINGS:
                        print('  OK  ' + source_label + ' (%.1fs, %.0f KB)' %
                              (time.monotonic()-started_at, len(raw)/1024), flush=True)
                    return data
                except urllib.error.HTTPError as exc:
                    if exc.code in (404, 410):
                        raise PageMissing('Not published: ' + url) from exc
                    if exc.code not in (407, 429, 500, 502, 503, 504):
                        raise SourceError('HTTP ' + str(exc.code) + ': ' + url) from exc
                    direct_failure = 'HTTP ' + str(exc.code)
                except http.client.IncompleteRead as exc:
                    # Discard the partial bytes. Retrying must start a new GET.
                    direct_failure = ('Incomplete HTTP transfer ('
                                      + str(len(exc.partial)) + ' bytes in final interrupted read)')
                except http.client.HTTPException as exc:
                    direct_failure = 'Broken HTTP response: ' + str(exc)
                except (urllib.error.URLError, TimeoutError, OSError) as exc:
                    direct_failure = str(exc)
                    if 'Tunnel connection failed: 503' in direct_failure:
                        direct_failure = 'Corporate HTTPS proxy returned 503 while opening the tunnel'
                except (UnicodeError, LookupError) as exc:
                    raise SourceError('Unexpected source encoding: ' + url) from exc
                if attempt < retry_limit:
                    print('  RETRY ' + source_label + ' after ' + str(direct_failure)[:100]
                          + ' (' + str(attempt + 1) + '/' + str(retry_limit) + ')', flush=True)
                    time.sleep(NETWORK_RETRY_PAUSE_SECONDS * attempt)

        if SHOW_DOWNLOAD_TIMINGS:
            print('  FAILED ' + source_label + ' after %.1fs' %
                  (time.monotonic() - started_at), flush=True)

        if NETWORK_MODE == 'urllib' or os.name != 'nt':
            raise SourceNetworkError(
                'Unable to download ' + url + '\n'
                + 'Python/company proxy: ' + str(direct_failure) + '\n'
                + 'No data were substituted or assumed missing.'
            )

        # Optional legacy fallback; not recommended in corporate PowerShell
        # Constrained Language mode. No bypass of corporate policy is attempted.
        try:
            if not self.windows_proxy_worked:
                print('Python could not reach this source. Trying Windows proxy fallback...')
            raw = _download_via_windows_proxy(url, WINDOWS_PROXY_TIMEOUT_SECONDS)
            data = _decode_source(url, raw)
            self.cache[url] = data
            self.windows_proxy_worked = True
            return data
        except PageMissing:
            raise
        except SourceError as exc:
            raise SourceNetworkError(
                'Unable to download ' + url + '\n'
                + 'Python/company proxy: ' + str(direct_failure) + '\n'
                + 'Windows fallback: ' + str(exc)
            ) from exc
        except (UnicodeError, LookupError) as exc:
            raise SourceError('Unexpected source encoding: ' + url) from exc


class SourceError(RuntimeError):
    """An unavailable source or unexpected source format; never means no changes."""


class PageMissing(SourceError):
    pass


class SourceNetworkError(SourceError):
    """A failed corporate network / proxy fetch, not a valid empty data set."""


def clean(value: object) -> str:
    return " ".join(str(value or "").split())


def stock_code(value: object) -> str:
    value = unicodedata.normalize("NFKC", clean(value))
    if not re.fullmatch(r"\d{1,5}(?:\.HK)?", value, re.I):
        raise SourceError(f"Invalid stock code: {value!r}")
    return value.split(".")[0].zfill(5)


def parse_date(value: object) -> date:
    value = clean(value)
    for fmt in ("%d/%m/%Y", "%Y/%m/%d", "%Y-%m-%d", "%Y%m%d"):
        try:
            return datetime.strptime(value, fmt).date()
        except ValueError:
            continue
    raise SourceError(f"Unrecognized source date: {value!r}")


def month_keys(day: date) -> list[str]:
    first = day.replace(day=1)
    following = (first + timedelta(days=32)).replace(day=1)
    third = (following + timedelta(days=32)).replace(day=1)
    return [first.strftime("%Y%m"), following.strftime("%Y%m"), third.strftime("%Y%m")]


def source_link(value: object, base: str) -> str:
    value = clean(value)
    if not value or value == "#":
        return ""
    url = urljoin(base, value)
    if urlparse(url).scheme not in ("http", "https"):
        raise SourceError(f"Unexpected source link: {value!r}")
    return url


@dataclass
class IPO:
    code: str
    name: str
    announcements: list[str]
    prospectuses: list[str]
    allotment_results: list[str]


@dataclass
class PHIP:
    key: str
    company: str
    board: str
    status: str
    posted: str
    title: str
    links: list[str]
    new: bool = False


@dataclass
class ShortList:
    effective: str
    url: str
    securities: dict[str, str]


@dataclass
class Lockup:
    code: str
    name: str
    expiry: str
    holder_type: str
    holders: str
    amount: str
    percent: str
    url: str
    price: str
    tab: str
    issue: str
    cumulative_change: str
    quote_url: str



def parse_ipos(document: str) -> list[IPO]:
    soup = parse_html(document, "html.parser")
    table = None
    for candidate in soup.select("table"):
        headers = clean(candidate.find("thead").get_text(" ")) if candidate.find("thead") else ""
        if "股份代號" in headers and "招股章程" in headers:
            table = candidate
            break
    if table is None:
        raise SourceError("Main Board IPO table was not found; the page may have changed.")
    rows: dict[str, IPO] = {}
    for tr in table.select("tbody tr"):
        cells = tr.find_all("td", recursive=False)
        if not cells:
            continue
        if (len(cells) == 1 and cells[0].get("colspan") == "5"
                and re.search(r"沒有|暫無|無資料|No\s+(?:records?|data)", clean(tr.get_text(" ")), re.I)):
            continue
        if len(cells) != 5:
            raise SourceError("Unexpected number of Main Board IPO columns.")
        if not clean(cells[0].get_text()) and ("沒有" in tr.get_text() or "No record" in tr.get_text()):
            continue
        code = stock_code(cells[0].get_text())
        links = [[source_link(a["href"], MAIN_BOARD_URL) for a in cell.select("a[href]")]
                 for cell in cells[2:5]]
        rows[code] = IPO(code, clean(cells[1].get_text(" ")), *links)
    # A recognizable empty table is valid; missing tables and malformed rows are errors.
    return list(rows.values())


def parse_phip(payload: dict, board: str, report_day: date) -> list[PHIP]:
    if not isinstance(payload, dict) or not isinstance(payload.get("app"), list):
        raise SourceError("Unexpected HKEX PHIP JSON schema (expected app array).")
    start = report_day - timedelta(days=6)
    rows: dict[str, PHIP] = {}
    statuses = {"A": "Active", "LP": "Lapsed", "RJ": "Rejected", "W": "Withdrawn", "LT": "Listed"}
    for applicant in payload["app"]:
        if not isinstance(applicant, dict) or "a" not in applicant or "id" not in applicant:
            raise SourceError("Unexpected PHIP applicant schema.")
        # Use each document's date, never the applicant's overall latest posting date.
        for bucket in ("ls", "ps"):
            proofs = applicant.get(bucket, [])
            if not isinstance(proofs, list):
                raise SourceError("Unexpected PHIP submission schema.")
            for proof in proofs:
                if not isinstance(proof, dict):
                    raise SourceError("Unexpected PHIP proof record (not an object).")
                title = clean(proof.get("nF", ""))
                if not ("聆訊後資料集" in title or re.search(r"\bPHIP\b|Post[- ]Hearing", title, re.I)):
                    continue
                posted = parse_date(proof.get("d", ""))
                if not start <= posted <= report_day:
                    continue
                links = sorted({url for field in ("u1", "u2")
                                if (url := source_link(proof.get(field, ""), PHIP_DOCUMENT_BASE))})
                identity = [board, str(applicant["id"]), posted.isoformat(), title, links]
                key = hashlib.sha256(json.dumps(identity, ensure_ascii=False).encode()).hexdigest()
                rows[key] = PHIP(key, clean(applicant["a"]), board,
                                 statuses.get(applicant.get("s"), clean(applicant.get("s"))),
                                 posted.isoformat(), title, links)
    return list(rows.values())


def fetch_phip(client: Client, report_day: date, include_gem: bool) -> list[PHIP]:
    rows: dict[str, PHIP] = {}
    boards = [("sehk", "Main Board")] + ([("gem", "GEM")] if include_gem else [])
    for slug, board in boards:
        # Active feeds retain PHIP history; inactive feeds capture recent status changes.
        # Listed feeds currently have no document history, but are checked as well.
        for stem in ("appactive_appphip", "appinactive", "applisted"):
            url = PHIP_FEED_BASE + f"{stem}_{slug}_c.json"
            try:
                payload = json.loads(client.text(url))
            except json.JSONDecodeError as exc:
                raise SourceError(f"Invalid PHIP JSON: {url}") from exc
            for row in parse_phip(payload, board, report_day):
                rows[row.key] = row
    return sorted(rows.values(), key=lambda row: (row.posted, row.company), reverse=True)


def parse_short_list(document: str, url: str, as_of: date) -> ShortList:
    if not re.search(r"</table\s*>", document, re.I) or not re.search(r"</html\s*>", document, re.I):
        raise SourceError(f"Incomplete short-selling HTML document: {url}")
    soup = parse_html(document, "html.parser")
    content = clean(soup.get_text(" "))
    match = re.search(r"Effective\s+Date\s*:\s*(\d{1,2}/\d{1,2}/\d{4})", content, re.I)
    if not match:
        raise SourceError(f"Short-selling effective date not found: {url}")
    effective = parse_date(match[1])
    if effective > as_of:
        raise SourceError(f"Future-effective short-selling list: {effective}")
    url_date = re.search(r"ds(\d{8})\.htm", url)
    if url_date and effective != parse_date(url_date[1]):
        raise SourceError(f"Short-list date does not match the requested URL: {url}")
    table = None
    for candidate in soup.select("table"):
        first = candidate.find("tr")
        if first and "Stock Code" in first.get_text(" ") and "Stock Short Name" in first.get_text(" "):
            table = candidate
            break
    if table is None:
        raise SourceError("HKEX short-selling securities table not found.")
    securities: dict[str, str] = {}
    numbers: list[int] = []
    for tr in table.select("tr")[1:]:
        cells = tr.find_all(["td", "th"], recursive=False)
        if not cells:
            continue
        if len(cells) < 3:
            raise SourceError("Malformed short-selling row.")
        number, raw_code, name = [clean(c.get_text(" ")) for c in cells[:3]]
        if number == "No." and raw_code == "Stock Code":
            continue
        if not number.isdigit() or not name:
            raise SourceError("Malformed short-selling row values.")
        code = stock_code(raw_code)
        if code in securities:
            raise SourceError(f"Duplicate short-selling code: {code}")
        numbers.append(int(number))
        securities[code] = name
    if not securities or numbers != list(range(1, len(numbers) + 1)):
        raise SourceError("Short-selling list is empty or incomplete (row sequence check failed).")
    return ShortList(effective.isoformat(), url, securities)


def find_short_list(client: Client, as_of: date, lookback: int) -> ShortList:
    for offset in range(lookback):
        candidate = as_of - timedelta(days=offset)
        url = SHORT_URL.format(day=candidate)
        try:
            return parse_short_list(client.text(url), url, as_of)
        except PageMissing:
            continue
        # A blocked/invalid page must fail, not silently skip to an older list.
    raise SourceError(f"No published short-selling list found in {lookback} days ending {as_of}.")


def short_changes(current: ShortList, previous: ShortList | None) -> tuple[dict, dict]:
    if previous is None:
        return {}, {}
    added = {code: current.securities[code] for code in sorted(current.securities.keys() - previous.securities.keys())}
    removed = {code: previous.securities[code] for code in sorted(previous.securities.keys() - current.securities.keys())}
    return added, removed


def short_history_changes(snapshots: list[ShortList]) -> list[dict]:
    """Keep every daily transition, including removals followed by re-additions."""
    ordered = sorted(snapshots, key=lambda item: item.effective)
    if len({item.effective for item in ordered}) != len(ordered):
        raise SourceError("Duplicate effective dates in short-selling history.")
    events = []
    for previous, current in zip(ordered, ordered[1:]):
        added, removed = short_changes(current, previous)
        for action, records in [("ADDED", added), ("REMOVED", removed)]:
            for code, name in records.items():
                events.append({"effective": current.effective, "change": action,
                               "code": code, "name": name, "url": current.url,
                               "previous_effective": previous.effective})
    return sorted(events, key=lambda event: (event["effective"], event["change"], event["code"]), reverse=True)


def recent_trading_dates(document: str, day: date, days: int = 5) -> list[date]:
    """Use the live government general-holiday calendar, including year boundaries."""
    try:
        payload = json.loads(document)
        calendars = payload["vcalendar"]
        holidays = {parse_date(event["dtstart"][0]) for calendar in calendars
                    for event in calendar["vevent"]}
    except (ValueError, TypeError, KeyError, IndexError) as exc:
        raise SourceError("Hong Kong holiday calendar changed format.") from exc
    by_year = {}
    for holiday in holidays:
        by_year.setdefault(holiday.year, set()).add(holiday)
    # Observed holidays may replace 1 January or 25 December when they fall
    # on Sunday. Check full-year coverage rather than requiring those dates.
    covered_years = {year for year, dates in by_year.items()
                     if len(dates) >= 16 and min(dates).month <= 3 and max(dates).month == 12}
    dates = []
    candidate = day
    while len(dates) < days:
        if candidate.year not in covered_years:
            raise SourceError(f"Holiday calendar does not cover {candidate.year}.")
        if candidate.weekday() < 5 and candidate not in holidays:
            dates.append(candidate)
        candidate -= timedelta(days=1)
    return list(reversed(dates))


def fetch_short_history(client: Client, day: date, start: date, lookback: int = 45) -> list[ShortList]:
    """Fetch every dated revision in the window and the preceding effective list.

    Only a confirmed missing page is skipped. A network or parsing failure never
    substitutes an older list or silently hides a daily eligibility change.
    """
    if start > day:
        raise ValueError("Comparison start must not be after the report date.")
    snapshots = []
    for offset in range(lookback):
        candidate = day - timedelta(days=offset)
        url = SHORT_URL.format(day=candidate)
        try:
            item = parse_short_list(client.text(url), url, day)
        except PageMissing:
            continue
        snapshots.append(item)
        if candidate < start:
            return list(reversed(snapshots))
    raise SourceError(f"No short-selling comparison baseline before {start} within {lookback} days; draft withheld.")


def embedded_lockups(document):
    match = re.search(r"\b(?:var|let|const)\s+freefloat_search_card_array\s*=\s*", document)
    if not match: return None
    try:
        payload, _ = json.JSONDecoder().raw_decode(document[match.end():])
        if not isinstance(payload, dict): raise ValueError("Expected object")
        return [row for dates in payload.values() for records in dates.values() for row in records]
    except (ValueError, TypeError, AttributeError) as exc:
        raise SourceError("etnet verification data changed format.") from exc


def decimal_value(text, percent=False):
    match = re.fullmatch(r"\s*([+-]?[\d,]+(?:\.\d+)?)\s*%?\s*", str(text))
    if not match: return None
    try: return Decimal(match[1].replace(",", ""))
    except InvalidOperation: return None


def hkd_millions(raw):
    if raw in ("", "--", "—"): return "—"
    match = re.fullmatch(r"(?:HKD|HK\$)\s*([\d,]+(?:\.\d+)?)\s*(億|萬)?", raw)
    if not match: return raw
    amount = Decimal(match[1].replace(",", ""))
    multiplier = {"億": Decimal(100), "萬": Decimal("0.01"), None: Decimal("0.000001")}[match[2]]
    return f"{amount * multiplier:,.1f}"


def parse_lockups(document, month):
    """Read visible monthly cards; reconcile counts and values to embedded JSON."""
    soup = parse_html(document)
    expected = None
    for anchor in soup.select("a[data-yearmonth]"):
        if anchor.get("data-yearmonth") == month:
            quantity = anchor.select_one(".quantity")
            if quantity and clean(quantity.get_text()).isdigit():
                expected = int(clean(quantity.get_text()))
                break
    rows = {}
    for card in soup.select(".section-card"):
        fields = {}
        for header in card.select(".card-header"):
            value = header.parent.select_one(".card-field")
            if value: fields[clean(header.get_text(" "))] = clean(value.get_text(" "))
        if "解禁日" not in fields: continue
        expiry = parse_date(fields["解禁日"])
        if expiry.strftime("%Y%m") != month: continue
        heading = card.select_one(".card-title h3")
        anchor = card.select_one('.card-title a[href*="ipo-info.php"]')
        match = re.fullmatch(r"(\d{1,5})\s+(.+)", clean(heading.get_text(" ")) if heading else "")
        if not match or not anchor: raise SourceError("A lock-up card is missing its company or code.")
        code = stock_code(match[1])
        url = source_link(anchor["href"], "https://www.etnet.com.hk/")
        query = parse_qs(urlparse(url).query)
        if stock_code(query.get("code", [""])[0]) != code:
            raise SourceError(f"Lock-up stock code/link mismatch: {code}")
        tab = query.get("tab", [""])[0]
        labels = {"cornerstone": ("Cornerstone", "基石投資者總數"),
                  "majorsharesholder": ("Major shareholder", "大股東總數")}
        if tab not in labels: raise SourceError(f"Unknown lock-up investor tab: {code}")
        holder_type, holder_label = labels[tab]
        if holder_label not in fields: raise SourceError(f"Lock-up investor type mismatch: {code}")
        for required in ["解禁總金額", "佔已發行股數", "按盤價", "累積升跌"]:
            if required not in fields: raise SourceError(f"Missing {required} for {code}")
        nominal = card.select_one(".nominal")
        price = clean(nominal.get_text()) if nominal else clean(fields["按盤價"].split()[0])
        quote = next((anchor for anchor in card.select('a[href*="realtime/quote.php"]')
                      if clean(anchor.get_text()) == "即時報價"), None)
        if not quote: raise SourceError(f"Missing quote link for {code}")
        quote_url = source_link(quote["href"], "https://www.etnet.com.hk/")
        if stock_code(parse_qs(urlparse(quote_url).query).get("code", [""])[0]) != code:
            raise SourceError(f"Lock-up quote link/code mismatch: {code}")
        percent = fields["佔已發行股數"]
        amount = fields["解禁總金額"]
        problems = []
        pct = decimal_value(percent)
        if pct is None:
            problems.append("Issued-share percentage unavailable")
        elif not Decimal(0) <= pct <= Decimal(100):
            problems.append(f"Source reports {percent} of issued shares")
        if amount in ("", "--", "—"): problems.append("Unlock value unavailable")
        row = Lockup(code, match[2], expiry.isoformat(), holder_type, fields[holder_label],
                     amount, percent, url, price, tab, "; ".join(problems),
                     fields["累積升跌"], quote_url)
        key = (code, row.expiry, tab)
        if key in rows: raise SourceError(f"Duplicate visible lock-up row: {key}")
        rows[key] = row
    if expected is None:
        raise SourceError(f"etnet monthly row-count control missing for {month}.")
    if len(rows) != expected:
        raise SourceError(f"etnet {month}: page says {expected} records; extracted {len(rows)}. Draft withheld.")
    embedded = embedded_lockups(document)
    if embedded is None:
        raise SourceError("etnet verification data is missing; draft withheld.")
    checks = {}
    for item in embedded:
        def value(key): return clean(item.get(key, {}).get("data", ""))
        expiry = parse_date(value("freefloatdate"))
        if expiry.strftime("%Y%m") != month: continue
        tab = value("freefloat_type")
        code = stock_code(value("stockcode"))
        key = (code, expiry.isoformat(), tab)
        comparison = (value("name"), value("totalcornerstonescnt" if tab == "cornerstone" else "totalmajorsharesholderscnt"),
                      value("currency_totalfreefloatamount"), value("totalpercentofissuedshares"), value("nominal"),
                      value("totalchg_percent"), source_link(item.get("stockcode", {}).get("quotelink", ""), "https://www.etnet.com.hk/"))
        if key in checks and checks[key] != comparison:
            raise SourceError(f"Conflicting etnet verification records: {key}")
        checks[key] = comparison
    if set(rows) != set(checks):
        raise SourceError(f"etnet {month}: visible and embedded lock-up records disagree. Draft withheld.")
    for key, row in rows.items():
        comparison = (row.name, row.holders, row.amount, row.percent, row.price, row.cumulative_change, row.quote_url)
        if comparison != checks[key]:
            raise SourceError(f"etnet values disagree for {row.code} / {row.holder_type}. Draft withheld.")
    return list(rows.values())


class BloombergError(SourceError):
    """Bloomberg data could not be retrieved or validated."""


def bloomberg_security(code):
    return str(int(stock_code(code))) + ' HK Equity'


def unlock_amount_hkd(raw):
    """Parse the source amount without rounding or assuming a missing currency.

    etnet commonly uses HKD/HK$ plus 萬 or 億. A foreign-currency or unknown
    format is flagged, never silently treated as HKD or zero.
    """
    value = re.sub(r'\s+', '', unicodedata.normalize('NFKC', str(raw)))
    match = re.fullmatch(
        r'(?:HKD|HK\$|港元)(\d{1,3}(?:,\d{3})+(?:\.\d+)?|\d+(?:\.\d+)?)'
        r'(百萬|百万|億|亿|萬|万|千)?', value, re.I)
    if not match:
        raise ValueError('Unlock amount is missing, non-HKD, or has an unsupported format: ' + str(raw))
    scale = {'': 1, '千': 1000, '萬': 10000, '万': 10000,
             '百萬': 1000000, '百万': 1000000, '億': 100000000, '亿': 100000000}
    return Decimal(match[1].replace(',', '')) * scale[match[2] or '']


def finite_decimal(value):
    if value is None or isinstance(value, bool):
        return None
    try:
        number = Decimal(str(value))
        return number if number.is_finite() else None
    except (InvalidOperation, ValueError):
        return None


def bloomberg_request_specs():
    end_date = str(BLOOMBERG_TURNOVER_END_DATE).strip()
    turnover_overrides = {'MARKET_DATA_OVERRIDE': 'TURNOVER', 'CALC_INTERVAL': '20D', 'CRNCY': 'HKD'}
    if end_date:
        if not re.fullmatch(r'\d{8}', end_date):
            raise BloombergError('BLOOMBERG_TURNOVER_END_DATE must be blank or YYYYMMDD.')
        try:
            parsed = datetime.strptime(end_date, '%Y%m%d').date()
        except ValueError as exc:
            raise BloombergError('Invalid Bloomberg turnover end date: ' + end_date) from exc
        if parsed > datetime.now(HK).date():
            raise BloombergError('Bloomberg turnover end date cannot be in the future.')
        turnover_overrides['END_DATE_OVERRIDE'] = end_date
    # Separate requests preserve the overrides of each supplied Excel formula.
    # Keep the supplied base BDP formula: market cap is in HKD millions.
    # Convert it explicitly in Python; do not assume an Excel scaling option
    # is supported as a ReferenceDataRequest field override.
    return [
        ('CRNCY_ADJ_MKT_CAP', {'EQY_FUND_CRNCY': 'HKD'}),
        ('EQY_FREE_FLOAT_PCT', {}),
        ('INTERVAL_AVG', turnover_overrides),
    ]


class BloombergClient:
    """Synchronous Desktop API requests with bounded waits and error auditing.

    API reference: https://bloomberg.github.io/blpapi-docs/python/3.26.9/
    Field names and override choices follow the supplied Excel formulas.
    """
    def __enter__(self):
        self.session = None
        try:
            import blpapi
        except (ImportError, OSError) as exc:
            raise BloombergError('blpapi is unavailable in this Python environment. '
                                 'Install Bloomberg\'s supported package and sign in to the desktop Terminal.') from exc
        self.api = blpapi
        try:
            options = blpapi.SessionOptions()
            options.setServerHost(BLOOMBERG_HOST)
            options.setServerPort(BLOOMBERG_PORT)
            options.setClientMode(blpapi.SessionOptions.DAPI)
            options.setConnectTimeout(BLOOMBERG_CONNECT_TIMEOUT_MS)
            options.setNumStartAttempts(1)
            options.setServiceCheckTimeout(BLOOMBERG_CONNECT_TIMEOUT_MS)
            options.setServiceDownloadTimeout(BLOOMBERG_CONNECT_TIMEOUT_MS)
            self.session = blpapi.Session(options)
            if not self.session.start():
                raise BloombergError('Cannot connect to Bloomberg Desktop API. Check that the Terminal is logged in.')
            if not self.session.openService('//blp/refdata'):
                raise BloombergError('Bloomberg //blp/refdata service is unavailable.')
            self.service = self.session.getService('//blp/refdata')
            return self
        except Exception:
            self.__exit__(None, None, None)
            raise

    def __exit__(self, *_):
        if self.session is not None:
            try:
                self.session.stop()
            except Exception as exc:
                LOG.warning('Bloomberg session cleanup failed: %s', exc)

    def reference(self, securities, field, overrides):
        request = self.service.createRequest('ReferenceDataRequest')
        for security in securities:
            request.getElement('securities').appendValue(security)
        request.getElement('fields').appendValue(field)
        for key, value in overrides.items():
            override = request.getElement('overrides').appendElement()
            override.setElement('fieldId', key)
            override.setElement('value', str(value))
        queue = self.api.EventQueue()
        correlation = self.session.sendRequest(request, eventQueue=queue)
        deadline = time.monotonic() + BLOOMBERG_REQUEST_TIMEOUT_SECONDS
        result, completed = {}, False
        try:
            while True:
                remaining = deadline - time.monotonic()
                if remaining <= 0:
                    raise BloombergError(field + ': request timed out; partial results discarded.')
                event = queue.nextEvent(max(1, min(500, int(remaining * 1000))))
                event_type = event.eventType()
                for message in event:
                    if event_type == self.api.Event.REQUEST_STATUS:
                        raise BloombergError(field + ': ' + str(message)[:600])
                    if event_type not in (self.api.Event.PARTIAL_RESPONSE, self.api.Event.RESPONSE):
                        continue
                    if message.hasElement('responseError'):
                        raise BloombergError(field + ': ' + str(message.getElement('responseError'))[:600])
                    if not message.hasElement('securityData'):
                        raise BloombergError(field + ': response has no securityData.')
                    data = message.getElement('securityData')
                    for index in range(data.numValues()):
                        item = data.getValueAsElement(index)
                        security = item.getElementAsString('security')
                        # sequenceNumber identifies the requested security even if Bloomberg
                        # returns a canonical name different from the input ticker.
                        if item.hasElement('sequenceNumber'):
                            sequence = item.getElementAsInteger('sequenceNumber')
                            if not 0 <= sequence < len(securities):
                                raise BloombergError('Unexpected Bloomberg security sequence number.')
                            security = securities[sequence]
                        if security not in securities:
                            raise BloombergError('Unexpected Bloomberg security: ' + security)
                        if item.hasElement('securityError'):
                            result[security] = (None, str(item.getElement('securityError'))[:600])
                            continue
                        error = None
                        if item.hasElement('fieldExceptions'):
                            exceptions = item.getElement('fieldExceptions')
                            for j in range(exceptions.numValues()):
                                exception = exceptions.getValueAsElement(j)
                                if exception.getElementAsString('fieldId') == field:
                                    error = str(exception.getElement('errorInfo'))[:600]
                        number = None
                        if not error and item.hasElement('fieldData'):
                            fields = item.getElement('fieldData')
                            if fields.hasElement(field):
                                element = fields.getElement(field)
                                if not element.isNull() and element.numValues():
                                    number = finite_decimal(element.getValueAsString())
                        result[security] = (number, error or (None if number is not None else 'Missing or non-finite value'))
                if event_type == self.api.Event.RESPONSE:
                    completed = True
                    break
            return {security: result.get(security, (None, 'Security omitted from response'))
                    for security in securities}
        finally:
            if not completed:
                try:
                    self.session.cancel(correlation)
                except Exception as exc:
                    LOG.warning('Bloomberg request cleanup failed: %s', exc)


def fetch_bloomberg_metrics(codes):
    securities = sorted({bloomberg_security(code) for code in codes})
    metrics = {security: {'fields': {}, 'errors': {}} for security in securities}
    specs = bloomberg_request_specs()
    audit = {'enabled': BLOOMBERG_ENABLED, 'requested_securities': len(securities),
             'started_at': datetime.now(HK).isoformat(), 'currency': 'HKD',
             'monetary_units': 'normalized amounts in absolute HKD',
             'raw_field_units': {'CRNCY_ADJ_MKT_CAP': 'HKD millions',
                                 'EQY_FREE_FLOAT_PCT': 'percent', 'INTERVAL_AVG': 'absolute HKD'},
             'unit_basis': 'Units follow the supplied Excel field definitions; validate with Terminal FLDS.',
             'turnover_end_date': str(BLOOMBERG_TURNOVER_END_DATE).strip() or None,
             'requests': [{'field': field, 'overrides': overrides} for field, overrides in specs]}
    if securities:
        if BLOOMBERG_BATCH_SIZE < 1 or BLOOMBERG_REQUEST_TIMEOUT_SECONDS <= 0:
            raise BloombergError('Bloomberg batch size and request timeout must be positive.')
        try:
            if not BLOOMBERG_ENABLED:
                raise BloombergError('Bloomberg enrichment disabled in settings.')
            print('Fetching Bloomberg lock-up metrics for ' + str(len(securities)) + ' unique securities ...', flush=True)
            with BloombergClient() as client:
                for start in range(0, len(securities), BLOOMBERG_BATCH_SIZE):
                    batch = securities[start:start + BLOOMBERG_BATCH_SIZE]
                    for field, overrides in specs:
                        try:
                            values = client.reference(batch, field, overrides)
                        except Exception as exc:
                            values = {security: (None, str(exc)[:600]) for security in batch}
                        for security, (value, error) in values.items():
                            if error:
                                metrics[security]['errors'][field] = error
                            else:
                                # Strings retain decimal precision and serialize safely to JSON.
                                metrics[security]['fields'][field] = str(value)
        except Exception as exc:
            audit['error'] = str(exc)[:600]
            for entry in metrics.values():
                for field, _ in specs:
                    if field not in entry['fields']:
                        entry['errors'].setdefault(field, audit['error'])
    audit['completed_at'] = datetime.now(HK).isoformat()
    audit['securities'] = metrics
    return metrics, audit


def lockup_ratios(raw_amount, fields, errors=None):
    """Calculate each tranche independently, using unrounded absolute HKD."""
    errors = errors or {}
    issues = []
    try:
        amount = unlock_amount_hkd(raw_amount)
    except ValueError as exc:
        amount = None
        issues.append(str(exc))

    def value(field, valid, description):
        number = finite_decimal(fields.get(field))
        if field in errors or number is None or not valid(number):
            issues.append(description + ': ' + errors.get(field, 'missing or outside the allowed range'))
            return None
        return number

    cap_millions = value('CRNCY_ADJ_MKT_CAP', lambda n: n > 0, 'Market cap unavailable')
    cap = cap_millions * Decimal(1000000) if cap_millions is not None else None
    free_float = value('EQY_FREE_FLOAT_PCT', lambda n: 0 < n <= 100, 'Free-float percentage unavailable')
    turnover = value('INTERVAL_AVG', lambda n: n > 0, '20D average turnover unavailable')
    float_cap = cap * free_float / Decimal(100) if cap is not None and free_float is not None else None
    percentage = amount / float_cap * Decimal(100) if amount is not None and float_cap is not None else None
    multiple = amount / turnover if amount is not None and turnover is not None else None
    values = {'unlock_amount_hkd': amount, 'market_cap_hkd': cap, 'free_float_pct': free_float,
              'float_market_cap_hkd': float_cap, 'avg_turnover_20d_hkd': turnover,
              'unlock_to_float_market_cap_pct': percentage, 'unlock_to_20d_avg_turnover_x': multiple}
    return {key: str(number) if number is not None else None for key, number in values.items()}, issues


def enrich_lockup_rows(rows):
    metrics, audit = fetch_bloomberg_metrics(row['code'] for row in rows)
    missing = 0
    for row in rows:
        security = bloomberg_security(row['code'])
        entry = metrics[security]
        values, issues = lockup_ratios(row['amount'], entry['fields'], entry['errors'])
        row.update(values)
        row['bloomberg_security'] = security
        row['bloomberg_issues'] = issues
        row['source_issue'] = row['issue']
        if issues:
            missing += 1
            row['issue'] = '; '.join(filter(None, [row['issue'], 'Bloomberg ratios: ' + '; '.join(issues)]))
    audit['tranches_with_unavailable_ratios'] = missing
    audit['status'] = 'no_lockups' if not rows else ('incomplete' if missing else 'ok')
    if missing and BLOOMBERG_REQUIRED:
        raise BloombergError(f'Bloomberg ratios unavailable for {missing} lock-up tranches; draft withheld. '
                             + next(row['issue'] for row in rows if row['bloomberg_issues']))
    return audit


def format_bloomberg_metric(row, field, suffix='', divisor=1):
    number = finite_decimal(row.get(field))
    return 'N/A' if number is None else f'{number / Decimal(divisor):,.2f}{suffix}'


def lockup_review_summary(row):
    """Keep the email actionable; raw API error messages stay in the JSON audit."""
    reasons = [row.get('source_issue', row.get('issue', ''))]
    if row.get('bloomberg_issues'):
        if row.get('unlock_amount_hkd') is None:
            reasons.append('Unlock amount cannot be read as HKD')
        if row.get('float_market_cap_hkd') is None:
            reasons.append('Free-float market cap unavailable')
        if row.get('avg_turnover_20d_hkd') is None:
            reasons.append('20D average turnover unavailable')
    return '; '.join(dict.fromkeys(reason for reason in reasons if reason))


SECTION_NAMES = {'ipo': 'IPO pipeline', 'phip': 'PHIP updates',
                 'short': 'Short-selling eligibility', 'lockup': 'Lock-up calendar'}


def section_snapshot(report):
    """Compare content, independent of ordering, refresh times and date labels.

    Rolling-window entries and exits count as changes. Market ratios compare at
    the precision displayed in the email. API diagnostics remain in the audit.
    """
    def records(rows, keys, fields):
        result = {}
        for row in rows:
            key = json.dumps([row[k] for k in keys], ensure_ascii=False)
            if key in result:
                raise SourceError('Duplicate record identity in section comparison: ' + key)
            result[key] = {field: sorted(row[field]) if isinstance(row[field], list) else row[field]
                           for field in fields}
        return result

    lockups = []
    for row in report['lockups']:
        item = {field: row[field] for field in ('code', 'expiry', 'holder_type', 'name', 'holders',
                'amount', 'percent', 'price', 'cumulative_change', 'url', 'quote_url', 'tab')}
        for field in ('float_market_cap_hkd', 'avg_turnover_20d_hkd'):
            item[field] = format_bloomberg_metric(row, field, divisor=1000000)
        for field in ('unlock_to_float_market_cap_pct', 'unlock_to_20d_avg_turnover_x'):
            item[field] = format_bloomberg_metric(row, field)
        item['eligibility'] = eligibility(row, report)[0]
        item['review'] = lockup_review_summary(row)
        lockups.append(item)
    return {'version': 1, 'sections': {
        'ipo': {'records': records(report['ipos'], ('code',),
                ('code', 'name', 'announcements', 'prospectuses', 'allotment_results')), 'context': {}},
        'phip': {'records': records(report['phip'], ('key',),
                ('company', 'board', 'status', 'posted', 'title', 'links')), 'context': {}},
        'short': {'records': {code: {'name': name} for code, name in report['short']['securities'].items()},
                  'context': {'events': records(report['short_changes'], ('effective', 'code', 'change'),
                                              ('effective', 'code', 'change', 'name', 'url'))}},
        'lockup': {'records': records(lockups, ('code', 'expiry', 'holder_type'),
                                     tuple(lockups[0]) if lockups else ()),
                   'context': {'turnover_end_date': report.get('bloomberg', {}).get('turnover_end_date')}}}}


def annotate_section_changes(report, previous):
    current = section_snapshot(report)
    compatible = previous is not None and previous['snapshot'].get('version') == current['version']
    report['comparison'] = {'basis': 'previous successfully saved draft',
                            'previous_generated_at': previous['generated_at'] if compatible else None,
                            'previous_day': previous['day'] if compatible else None}
    changes = {}
    for key, now in current['sections'].items():
        if not compatible or key not in previous['snapshot']['sections']:
            changes[key] = {'status': 'baseline', 'added': 0, 'removed': 0, 'updated': 0,
                            'context_changed': False, 'summary': 'Comparison starts with the next saved draft.'}
            continue
        before = previous['snapshot']['sections'][key]
        old_keys, new_keys = set(before['records']), set(now['records'])
        added, removed = new_keys - old_keys, old_keys - new_keys
        updated = {item for item in old_keys & new_keys if before['records'][item] != now['records'][item]}
        context_changed = before['context'] != now['context']
        parts = [str(len(items)) + ' ' + label for items, label in
                 ((added, 'added'), (removed, 'removed'), (updated, 'updated')) if items]
        if context_changed:
            parts.append('5-day event list updated' if key == 'short' else 'Turnover end date changed')
        changes[key] = {'status': 'changed' if parts else 'unchanged', 'added': len(added),
                        'removed': len(removed), 'updated': len(updated), 'context_changed': context_changed,
                        'summary': ' · '.join(parts) if parts else 'No content changes.'}
    report['section_changes'] = changes
    # PHIP NEW now means added since the same comparison draft as the headers.
    old_phips = previous['snapshot']['sections'].get('phip', {}).get('records', {}) if compatible else None
    for row in report['phip']:
        row['new'] = old_phips is not None and json.dumps([row['key']], ensure_ascii=False) not in old_phips
    return current


class State:
    def __init__(self, path: Path) -> None:
        path.parent.mkdir(parents=True, exist_ok=True)
        self.db = sqlite3.connect(path)
        self.db.execute("""CREATE TABLE IF NOT EXISTS runs (
            day TEXT NOT NULL, mode TEXT NOT NULL, short_json TEXT NOT NULL,
            phip_json TEXT NOT NULL, created_at TEXT NOT NULL, receipt_json TEXT,
            PRIMARY KEY(day, mode))""")
        self.db.execute("""CREATE TABLE IF NOT EXISTS section_snapshots (
            day TEXT NOT NULL, mode TEXT NOT NULL, generated_at TEXT NOT NULL,
            snapshot_json TEXT NOT NULL, PRIMARY KEY(day, mode))""")
        self.db.commit()

    def previous(self, day: date, mode: str) -> dict | None:
        row = self.db.execute("SELECT day, short_json, phip_json FROM runs WHERE day < ? AND mode = ? ORDER BY day DESC LIMIT 1",
                              (day.isoformat(), mode)).fetchone()
        if row is None:
            return None
        return {"day": row[0], "short": ShortList(**json.loads(row[1])), "phip": set(json.loads(row[2]))}

    def already_sent(self, day: date) -> bool:
        return self.db.execute("SELECT 1 FROM runs WHERE day=? AND mode='send'",
                               (day.isoformat(),)).fetchone() is not None

    def previous_sections(self, day: date, mode: str):
        row = self.db.execute('SELECT day, generated_at, snapshot_json FROM section_snapshots '
                              'WHERE day <= ? AND mode = ? ORDER BY day DESC LIMIT 1',
                              (day.isoformat(), mode)).fetchone()
        return {'day': row[0], 'generated_at': row[1], 'snapshot': json.loads(row[2])} if row else None

    def save(self, day: date, mode: str, short: ShortList, phip: list[PHIP], receipt: dict | None,
             sections=None, generated_at=None) -> None:
        with self.db:
            self.db.execute("INSERT OR REPLACE INTO runs VALUES (?, ?, ?, ?, ?, ?)",
                            (day.isoformat(), mode, json.dumps(asdict(short), ensure_ascii=False),
                             json.dumps([p.key for p in phip]), datetime.now(HK).isoformat(),
                             json.dumps(receipt) if receipt else None))
            if sections is not None:
                self.db.execute('INSERT OR REPLACE INTO section_snapshots VALUES (?, ?, ?, ?)',
                                (day.isoformat(), mode, generated_at or datetime.now(HK).isoformat(),
                                 json.dumps(sections, ensure_ascii=False, sort_keys=True)))

    def close(self) -> None:
        self.db.close()


@contextmanager
def run_lock(state_dir: Path):
    """Cross-platform process lock released automatically if the process exits."""
    state_dir.mkdir(parents=True, exist_ok=True)
    with (state_dir / "run.lock").open("a+b") as handle:
        try:
            if os.name == "nt":
                import msvcrt
                if os.fstat(handle.fileno()).st_size == 0:
                    handle.write(b"0"); handle.flush()
                handle.seek(0)
                msvcrt.locking(handle.fileno(), msvcrt.LK_NBLCK, 1)
            else:
                import fcntl
                fcntl.flock(handle.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
        except OSError as exc:
            raise RuntimeError("Another digest process is already running.") from exc
        try:
            yield
        finally:
            if os.name == "nt":
                handle.seek(0); msvcrt.locking(handle.fileno(), msvcrt.LK_UNLCK, 1)
            else:
                fcntl.flock(handle.fileno(), fcntl.LOCK_UN)




def e(value):
    return html.escape(str(value), quote=True)


def link(url, label):
    """Readable clickable hyperlinks with explicit Outlook-friendly font size."""
    label_html = e(label)
    if not url:
        return label_html
    return (f'<a href="{e(url)}" style="color:#0759A4;'
            'font-family:Arial,Microsoft JhengHei,sans-serif;'
            'font-size:16px;font-weight:700;text-decoration:underline;">'
            f'<u style="color:#0759A4;">{label_html}</u></a>')

def nice_date(value, year=False):
    return parse_date(value).strftime("%d %b %Y" if year else "%d %b")


def badge(label, color="neutral"):
    """High-contrast, large YES / NO / NEW / REVIEW indicator for Outlook."""
    colors = {
        "neutral": ("#e8edf2", "#213245", "#b6c1cc"),
        "green": ("#087544", "#ffffff", "#075c36"),
        "red": ("#a82032", "#ffffff", "#871929"),
        "blue": ("#1257b5", "#ffffff", "#0c468f"),
        "amber": ("#f4ce65", "#332700", "#a67b17"),
    }
    background, foreground, border = colors[color]
    return (f'<span style="display:inline-block;padding:6px 10px;'
            f'font-family:Arial,sans-serif;font-size:16px;line-height:22px;'
            f'font-weight:700;text-align:center;'
            f'background-color:{background};color:{foreground};border:1px solid {border};'
            f'white-space:nowrap;"><strong>{e(label)}</strong></span>')

def table(headers, rows, highlighted=None, widths=None):
    """Compact Word/Outlook-safe tables; optional explicit widths for lock-ups."""
    if not rows:
        return ('<p style="font-family:Arial,Microsoft JhengHei,sans-serif;'
                'font-size:16px;color:#495462;line-height:1.5;">No updates in this period.</p>')
    if widths is not None and (len(widths) != len(headers) or sum(widths) != 100):
        raise ValueError('Column widths must match headers and sum to 100%.')
    head = []
    for i, h in enumerate(headers):
        right = h.startswith('#')
        width_attr = f' width="{widths[i]}%"' if widths else ''
        width_style = f'width:{widths[i]}%;' if widths else ''
        head.append(
            f'<th scope="col"{width_attr} style="{width_style}text-align:{"right" if right else "left"};'
            'padding:12px 9px;background-color:#eaf0f6;border-bottom:2px solid #9eafc0;'
            'font-family:Arial,Microsoft JhengHei,sans-serif;'
            'font-size:14px;font-weight:700;color:#26394c;line-height:1.4;'
            f'white-space:normal;vertical-align:bottom;">{e(h.lstrip("#"))}</th>'
        )
    body = []
    for index, row in enumerate(rows):
        if len(row) != len(headers):
            raise ValueError('Report table column mismatch.')
        background = '#fff6e5' if highlighted and index in highlighted else ('#f7f9fc' if index % 2 else '#ffffff')
        cells = []
        for number, value in enumerate(row):
            align = 'right' if headers[number].startswith('#') else 'left'
            width_attr = f' width="{widths[number]}%"' if widths else ''
            width_style = f'width:{widths[number]}%;' if widths else ''
            cells.append(
                f'<td{width_attr} style="{width_style}text-align:{align};padding:13px 9px;'
                f'vertical-align:top;border-bottom:1px solid #d8e0e8;background-color:{background};'
                'font-family:Arial,Microsoft JhengHei,sans-serif;font-size:16px;'
                f'color:#263341;line-height:1.45;overflow-wrap:break-word;">{value}</td>'
            )
        body.append('<tr>' + ''.join(cells) + '</tr>')
    fixed = 'table-layout:fixed;' if widths else ''
    return ('<table role="presentation" width="100%" cellspacing="0" cellpadding="0" border="0" '
            f'style="width:100%;border-collapse:collapse;{fixed}"><thead><tr>'
            + ''.join(head) + '</tr></thead><tbody>' + ''.join(body) + '</tbody></table>')

def section_status(change):
    return {'changed': 'CHANGED', 'unchanged': 'NO CHANGE', 'baseline': 'BASELINE'}.get(
        (change or {}).get('status'), 'BASELINE')


def section(number, title, detail, change=None, review_count=0):
    """Large numbered headings, implemented as a table for Classic Outlook."""
    change = change or {'status': 'baseline', 'summary': 'Comparison starts with the next saved draft.'}
    changed = change['status'] == 'changed'
    status = ('<span style="display:inline-block;margin-left:12px;padding:4px 8px;'
              'font-family:Arial,sans-serif;font-size:12px;line-height:18px;font-weight:700;'
              f'background:{"#e2efff" if changed else "#eef1f4"};'
              f'color:{"#0759A4" if changed else "#526474"};">{section_status(change)}</span>')
    if review_count:
        status += ('<span style="display:inline-block;margin-left:7px;padding:4px 8px;font-size:12px;'
                   f'line-height:18px;font-weight:700;background:#fff1ca;color:#795b00;">REVIEW {review_count}</span>')
    summary = ('<div style="font-size:14px;color:#0759A4;margin-top:5px;">'
               + e(change['summary']) + '</div>') if changed else ''
    return (
        '<table role="presentation" width="100%" cellspacing="0" cellpadding="0" border="0" '
        'style="width:100%;border-collapse:collapse;margin-top:30px;margin-bottom:13px;'
        'border-top:2px solid #b41529;">'
        '<tr><td width="67" valign="top" style="width:67px;padding:16px 10px 8px 0;vertical-align:top;">'
        '<table role="presentation" cellspacing="0" cellpadding="0" border="0"><tr>'
        '<td bgcolor="#b41529" style="background-color:#b41529;color:white;'
        'font-family:Arial,sans-serif;text-align:center;padding:6px 11px;'
        f'font-size:22px;font-weight:700;line-height:29px;">{number:02d}</td>'
        '</tr></table></td>'
        '<td valign="top" style="padding:14px 0 10px;vertical-align:top;'
        'font-family:Calibri,Arial,Microsoft JhengHei,sans-serif;">'
        f'<div style="font-size:22px;font-weight:700;color:#172b3f;line-height:1.25;">{e(title)}{status}</div>'
        + summary +
        f'<div style="font-size:14px;color:#48596a;line-height:1.55;margin-top:5px;">{detail}</div>'
        '</td></tr></table>'
    )


def section_text_heading(report, key, title):
    change = report.get('section_changes', {}).get(key, {})
    return title + ' [' + section_status(change) + ']' + (
        ' — ' + change['summary'] if change.get('status') == 'changed' else '')


def eligibility(row, report):
    code = row['code']
    if code not in report['short']['securities']: return 'NO', 'red'
    if code in report['added']: return 'NEW', 'blue'
    return 'YES', 'green'


def email_subject(report):
    return EMAIL_SUBJECT.replace('{date}', parse_date(report['day']).strftime('%d %b %Y'))


def render_report(report):
    day = parse_date(report['day'])
    visible_lockups = report['lockups']  # Preserve flagged rows; mark for review instead of silently hiding
    subject = email_subject(report)
    short = report['short']
    window = report['short_window']
    events = report['short_changes']
    addition_count = sum(event['change'] == 'ADDED' for event in events)
    removal_count = sum(event['change'] == 'REMOVED' for event in events)
    text = [subject, '', f'Updated {datetime.fromisoformat(report["generated_at"]):%d %b %Y, %H:%M} HKT', '']
    fragments = []
    fragments.append('<table role="presentation" width="100%" cellspacing="0" cellpadding="0" style="background:#202a36;color:white;border-top:5px solid #b41529"><tr><td style="padding:25px 28px 26px">'
                     '<div style="font-size:30px;line-height:1.25;font-weight:700;letter-spacing:-.2px">IPO, PHIP &amp; Lock-up Update</div>'
                     f'<div style="font-size:15px;color:#e2e7ed;margin-top:12px">{day:%A, %d %B %Y} &nbsp; · &nbsp; Updated {datetime.fromisoformat(report["generated_at"]):%H:%M} HKT</div></td></tr></table>')
    stats = [(str(len(report['ipos'])), 'MAIN BOARD IPOs'), (str(len(report['phip'])), 'PHIPs · 7 DAYS'),
             (f'+{addition_count} / −{removal_count}', 'SHORT-LIST CHANGES · 5 DAYS'),
             (str(len(visible_lockups)), 'LOCK-UP TRANCHES / %d REVIEW' % sum(bool(r['issue']) for r in visible_lockups))]
    fragments.append('<table role="presentation" width="100%" cellspacing="0" cellpadding="0" style="border-bottom:1px solid #d9dde2"><tr>' +
                     ''.join(f'<td width="25%" style="padding:18px 18px 17px;border-right:1px solid #eceef1"><div style="font-size:28px;color:#202a36;font-weight:700;line-height:1.2">{e(value)}</div><div style="font-size:13px;color:#46596c;font-weight:700;letter-spacing:.1px;margin-top:7px">{label}</div></td>' for value, label in stats) + '</tr></table>')
    fragments.append('<div style="padding:0 28px 25px">')
    section_changes = report.get('section_changes', {})
    previous_time = report.get('comparison', {}).get('previous_generated_at')
    changed_names = [SECTION_NAMES[key] for key, change in section_changes.items() if change['status'] == 'changed']
    if previous_time:
        comparison_line = ('Compared with the saved draft from '
                           + datetime.fromisoformat(previous_time).strftime('%d %b %Y, %H:%M HKT') + '.')
        change_line = 'Changed sections: ' + ', '.join(changed_names) + '.' if changed_names else 'No section content changes.'
    else:
        comparison_line = 'Baseline saved on this run. Change flags begin with the next successfully saved draft.'
        change_line = 'Section comparison baseline'
    fragments.append('<p style="font-size:16px;line-height:1.5;color:#172b3f;margin:22px 0 5px;">'
                     + '<strong>' + e(change_line) + '</strong></p>'
                     + '<p style="font-size:13px;line-height:1.5;color:#526474;margin:0 0 10px;">'
                     + e(comparison_line) + ' Refresh times alone do not count. Added and removed refer to entries in the displayed lists.</p>')
    text.extend([change_line, comparison_line, ''])
    fragments.append(section(1, 'IPO pipeline', link(MAIN_BOARD_URL, 'Main Board'), section_changes.get('ipo')))
    ipo_rows = []
    text.append(section_text_heading(report, 'ipo', '1. IPO PIPELINE — MAIN BOARD'))
    for row in report['ipos']:
        documents = []
        for key, label in [('announcements', 'Announcement'), ('prospectuses', 'Prospectus'), ('allotment_results', 'Allotment')]:
            documents.extend(link(url, label + (f' {i}' if len(row[key]) > 1 else '')) for i, url in enumerate(row[key], 1))
        ipo_rows.append([f'<strong>{e(row["code"])}</strong>', e(row['name']), '<br>'.join(documents) or '—'])
        text.append(f'{row["code"]} | {row["name"]}')
        for key in ['announcements', 'prospectuses', 'allotment_results']:
            text.extend(f'  {key}: {url}' for url in row[key])
    fragments.append(table(['CODE', 'ISSUER', 'DOCUMENTS'], ipo_rows))
    phip_range = f'{nice_date(report["phip_start"])}–{nice_date(report["day"])} · past 7 calendar days'
    fragments.append(section(2, 'PHIP updates', e(phip_range) + ' &nbsp; · &nbsp; ' + link(PHIP_PAGE_URL, 'HKEX filing index'), section_changes.get('phip')))
    text.extend(['', section_text_heading(report, 'phip', '2. PHIP UPDATES'), phip_range])
    phip_rows, highlights = [], set()
    for row in report['phip']:
        new = badge('NEW', 'blue') + ' ' if row['new'] else ''
        if row['new']: highlights.add(len(phip_rows))
        # Each link comes from the verified HKEX filing feed; never fabricate missing URLs.
        filing_urls = row['links']
        filing_title = link(filing_urls[0], row['title']) if filing_urls else e(row['title'])
        issuer = link(filing_urls[0], row['company']) if filing_urls else e(row['company'])
        filing_actions = (' &nbsp; | &nbsp; '.join(
            link(url, 'Open filing' + (f' {i}' if len(filing_urls) > 1 else ''))
            for i, url in enumerate(filing_urls, 1)
        ) if filing_urls else '<span style="color:#815100;">No document URL in HKEX feed</span>')
        phip_rows.append([e(nice_date(row['posted'])), new + issuer, e(row['board']),
                          filing_title + '<div style="margin-top:8px;font-size:14px;">' + filing_actions + '</div>'])
        text.append(f'{row["posted"]} | {row["company"]} | {row["board"]} | {row["title"]}' + (' | NEW' if row['new'] else ''))
        text.extend(row['links'])
    fragments.append(table(['POSTED', 'ISSUER', 'BOARD', 'FILING'], phip_rows, highlights))
    if not phip_rows: text.append('No filings in the seven-day window.')
    comparison = f'{nice_date(window["start"], True)}–{nice_date(window["end"], True)} · {window["label"]}'
    detail = 'Latest effective list: ' + link(short['url'], nice_date(short['effective'], True))
    fragments.append(section(3, 'Short-selling eligibility', detail, section_changes.get('short')))
    fragments.append(f'<p style="font-size:14px;color:#48596a;margin:0 0 12px">{e(comparison)} &nbsp; · &nbsp; {len(short["securities"]):,} eligible securities.</p>')
    text.extend(['', section_text_heading(report, 'short', '3. SHORT-SELLING ELIGIBILITY'), comparison, 'Latest effective list: ' + short['effective']])
    changes = []
    for event in events:
        color = 'blue' if event['change'] == 'ADDED' else 'red'
        changes.append([link(event['url'], nice_date(event['effective'], True)), badge(event['change'], color),
                        '<strong>' + e(event['code']) + '</strong>', e(event['name'])])
        text.append(f'{event["effective"]} | {event["change"]} | {event["code"]} | {event["name"]}')
    if changes:
        fragments.append(table(['EFFECTIVE DATE', 'CHANGE', 'CODE', 'HKEX SHORT NAME'], changes))
        if not removal_count: fragments.append('<p style="font-size:14px;color:#48596a;margin:8px 0 0">Removals: none.</p>')
    else:
        fragments.append('<p style="font-size:15px">No additions or removals in this period.</p>')
        text.append('No additions or removals in this period.')
    expiry_scope = 'Includes earlier expiries this month' if report.get('include_expired') else 'Upcoming expiries only'
    fragments.append(section(4, 'Lock-up calendar', f'{nice_date(report["day"])} through {datetime.strptime(report["months"][-1], "%Y%m"):%B %Y} &nbsp; · &nbsp; {expiry_scope}',
                             section_changes.get('lockup'), sum(bool(r['issue']) for r in visible_lockups)))
    if report.get('manual_sources'):
        manual_note = '; '.join(month + ': ' + when for month, when in sorted(report['manual_sources'].items()))
        fragments.append('<p style="font-size:14px;color:#805c00;background:#fff8e8;padding:10px 12px">'
                         '<strong>SOURCE REVIEW:</strong> etnet lock-up data include a browser-saved '
                         'page (' + e(manual_note) + '). Verify the source timestamps before distribution.</p>')
        text.extend(['SOURCE REVIEW: etnet browser-saved HTML (' + manual_note
                     + '). Verify source timestamps before distribution.'])
    if report.get('cached_sources'):
        cache_note = '; '.join(month + ': ' + when for month, when in sorted(report['cached_sources'].items()))
        fragments.append('<p style="font-size:14px;color:#805c00;background:#fff8e8;padding:10px 12px">'
                         '<strong>SOURCE REVIEW:</strong> Reused verified etnet HTML fetched earlier today ('
                         + e(cache_note) + '). Check its quote timestamps before distribution.</p>')
        text.append('SOURCE REVIEW: reused verified same-day etnet HTML (' + cache_note + '). Check quote timestamps.')
    fragments.append('<p style="font-family:Arial,Microsoft JhengHei,sans-serif;font-size:15px;line-height:2.5;color:#364b60;margin:0 0 16px">'
                     + badge('YES', 'green') + ' Can short &nbsp; &nbsp; '
                     + badge('NEW', 'blue') + ' Newly added to short-selling list &nbsp; &nbsp; '
                     + badge('NO', 'red') + ' Not shortable</p>')
    text.extend(['', section_text_heading(report, 'lockup', '4. LOCK-UP CALENDAR'), 'YES = shortable; NEW = added within the 5-day short-list window; NO = not shortable.'])
    bloomberg = report.get('bloomberg', {})
    ratio_note = ('Unlock / float cap (%) = unlock value ÷ free-float market cap × 100. '
                  'Unlock / 20D turnover (x) = unlock value ÷ Bloomberg 20D average turnover. '
                  'HKDm means HKD millions. ')
    timing_note = ('Bloomberg market cap and free float are current snapshots; '
                   'etnet unlock values use the quote times below. ')
    if bloomberg.get('turnover_end_date'):
        timing_note += ('The turnover interval ends on ' + bloomberg['turnover_end_date']
                       + '; this does not make the other inputs historical. ')
    else:
        timing_note += 'Turnover uses Bloomberg\'s latest 20D interval. '
    if bloomberg.get('completed_at'):
        timing_note += ('Bloomberg checked '
                       + datetime.fromisoformat(bloomberg['completed_at']).strftime('%d %b %Y %H:%M HKT') + '. ')
    if bloomberg.get('tranches_with_unavailable_ratios'):
        count = bloomberg['tranches_with_unavailable_ratios']
        ratio_note += str(count) + (' tranche needs' if count == 1 else ' tranches need') + ' review; unavailable ratios show N/A.'
    fragments.append('<p style="font-size:14px;line-height:1.6;color:#48596a;margin:0 0 16px">'
                     + e(ratio_note) + '</p>')
    text.append(ratio_note)
    last_added = {}
    for event in events:
        if event['change'] == 'ADDED':
            last_added[event['code']] = max(event['effective'], last_added.get(event['code'], ''))
    for month in report['months']:
        monthly = [r for r in visible_lockups if r['expiry'].replace('-', '').startswith(month)]
        label = datetime.strptime(month, '%Y%m').strftime('%B %Y').upper()
        month_url = LOCKUP_URL.format(month=month)
        fragments.append('<div style="margin:24px 0 10px;font-family:Arial,Microsoft JhengHei,sans-serif;'
                         'font-size:18px;font-weight:700;color:#25394b;">'
                         + e(label) + ' <span style="font-size:15px;font-weight:400;color:#526474;">'
                         + f' / {len(monthly)} tranches &nbsp; · &nbsp; ' + link(month_url, 'View etnet source')
                         + '</span></div>')
        rows, highlights = [], set()
        for row in monthly:
            days = (parse_date(row['expiry']) - day).days
            tag, color = eligibility(row, report)
            if tag == 'NEW' or row['issue']:
                highlights.add(len(rows))
            relative_date = 'Today' if days == 0 else (f'In {days} days' if days > 0 else f'{-days} days ago')
            date_cell = ('<strong style="font-size:16px;color:#172b3f;white-space:nowrap">'
                         + e(row['expiry'].replace('-', '/')) + '</strong>'
                         + '<div style="font-size:14px;color:#536578;margin-top:6px;">'
                         + e(relative_date) + '</div>')
            holder_label = '基石投資者總數' if row['tab'] == 'cornerstone' else '大股東總數'
            security = ('<strong style="font-size:16px;">'
                        + link(row['url'], row['code'] + ' ' + row['name']) + '</strong>'
                        + '<div style="font-size:14px;color:#4c5c6f;margin-top:7px;">'
                        + e(holder_label) + ': <strong>' + e(row['holders']) + '</strong></div>')
            unlock_display = ('HKDm ' + format_bloomberg_metric(row, 'unlock_amount_hkd', divisor=1000000)
                              if row.get('unlock_amount_hkd') is not None else row['amount'])
            unlock = ('<strong style="font-size:16px;color:#1f2e3f;">' + e(unlock_display) + '</strong>'
                      + '<div style="font-size:14px;color:#4c5c6f;margin-top:6px;">'
                      + '佔已發行股數: <strong>' + e(row['percent']) + '</strong></div>')
            float_ratio = format_bloomberg_metric(row, 'unlock_to_float_market_cap_pct', '%')
            turnover_ratio = format_bloomberg_metric(row, 'unlock_to_20d_avg_turnover_x', 'x')
            ratios = ('<div style="font-size:13px;color:#4c5c6f;">Unlock / float cap</div>'
                      + '<strong style="font-size:16px;color:#1f2e3f;">' + e(float_ratio) + '</strong>'
                      + '<div style="font-size:12px;color:#526474;">Float cap HKDm: '
                      + e(format_bloomberg_metric(row, 'float_market_cap_hkd', divisor=1000000)) + '</div>'
                      + '<div style="font-size:13px;color:#4c5c6f;margin-top:9px;">Unlock / 20D turnover</div>'
                      + '<strong style="font-size:16px;color:#1f2e3f;">' + e(turnover_ratio) + '</strong>'
                      + '<div style="font-size:12px;color:#526474;">20D avg HKDm: '
                      + e(format_bloomberg_metric(row, 'avg_turnover_20d_hkd', divisor=1000000)) + '</div>')
            change_parts = row['cumulative_change'].split(' ', 1)
            change_color = '#b11d2b' if row['cumulative_change'].startswith('-') else '#087544'
            change_value = ('<strong style="font-size:16px;color:' + change_color + ';">'
                            + '<br>'.join(e(part) for part in change_parts) + '</strong>')
            short_cell = badge(tag, color)
            if tag == 'NEW':
                short_cell += ('<div style="font-size:14px;line-height:1.5;color:#0759A4;margin-top:8px;">'
                               + 'Since ' + e(nice_date(last_added[row['code']])) + '</div>')
            if row['issue']:
                short_cell += '<div style="margin-top:8px;">' + badge('REVIEW', 'amber') + '</div>'
            rows.append([date_cell, security, unlock, ratios, change_value, short_cell])
            text.append(f'{row["expiry"]} | {row["code"]} {row["name"]} | {holder_label} {row["holders"]} | 解禁總金額 {row["amount"]} | 佔已發行股數 {row["percent"]} | 累積升跌 {row["cumulative_change"]} | Shortable: {tag}')
            text.append('  解禁金額 / Float market cap: ' + float_ratio
                        + ' | 解禁金額 / 20D average turnover: ' + turnover_ratio)
            text.append('  Bloomberg ' + row.get('bloomberg_security', bloomberg_security(row['code']))
                        + ' | Float market cap HKDm: ' + format_bloomberg_metric(row, 'float_market_cap_hkd', divisor=1000000)
                        + ' | 20D average turnover HKDm: ' + format_bloomberg_metric(row, 'avg_turnover_20d_hkd', divisor=1000000))
            if row['issue']:
                text.append('  MANUAL REVIEW: ' + lockup_review_summary(row))
            text.append('Source: ' + row['url'])
        fragments.append(table(['解禁日', '股票 / 投資者', '解禁金額 / 持股比例', '解禁金額比率', '累積升跌', 'SHORTABLE'],
                               rows, highlights, widths=[14, 22, 21, 20, 10, 13]))
        for row in monthly:
            if row['issue']:
                label = row['code'] + ' / ' + row['expiry'] + ' / ' + row['holder_type']
                fragments.append('<p style="font-size:13px;line-height:1.5;color:#795b00;margin:10px 0;">'
                                 + '<strong>REVIEW ' + e(label) + ':</strong> ' + e(lockup_review_summary(row)) + '</p>')
    quote_asof = ', '.join(f"{month}: {when} HKT" for month, when in report.get('source_asof', {}).items() if when)
    note = 'Compilation for manual review before distribution. HKEX: listing, PHIP and short-selling; etnet: lock-up expiry, value and issued-share percentages; Bloomberg: market cap, free-float percentage and 20-day average turnover.'
    note += ' ' + timing_note
    if quote_asof:
        note += ' etnet indicated update times: ' + quote_asof + '.'
    fragments.append('<p style="font-size:13px;line-height:1.6;color:#536578;margin:18px 0 8px;border-top:1px solid #e0e4e8;padding-top:14px">' + e(note) + '</p>')
    text.extend(['', note])
    fragments.append('</div>')
    content = ''.join(fragments)
    document = '<!doctype html><html lang="zh-Hant"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>' + e(subject) + '</title></head><body style="margin:0;background:#edf0f3;color:#222b36;font-family:Arial,\'Microsoft JhengHei\',sans-serif;font-size:16px;line-height:1.5"><div id="email-content" style="max-width:900px;margin:18px auto;background:white;color:#222b36;font-family:Arial,\'Microsoft JhengHei\',sans-serif;font-size:16px;line-height:1.5">' + content + '</div></body></html>'
    return document, '\n'.join(text) + '\n'


def make_email(report, html_body, text_body):
    message = EmailMessage(policy=SMTP)
    message['Subject'] = email_subject(report)
    if EMAIL_TO.strip():
        message['To'] = ', '.join(x.strip() for x in re.split(r'[,;]+', EMAIL_TO) if x.strip())
    if EMAIL_CC.strip():
        message['Cc'] = ', '.join(x.strip() for x in re.split(r'[,;]+', EMAIL_CC) if x.strip())
    message['Date'] = format_datetime(datetime.now(HK))
    message['Message-ID'] = make_msgid(domain='ipo-daily.local')
    message['X-Unsent'] = '1'
    message.set_content(text_body)
    message.add_alternative(html_body, subtype='html')
    return message


def preview_document(document, draft_name):
    toolbar = '<div style="max-width:900px;margin:20px auto 0;display:flex;gap:10px;align-items:center;font-family:Arial,sans-serif"><button onclick="copyEmail()" style="background:#263443;color:white;border:0;padding:11px 17px;cursor:pointer;font-weight:600">Copy formatted email</button><a href="' + e(draft_name) + '" download style="color:#263443;font-size:12px;text-decoration:none;border:1px solid #bac1c9;padding:10px 15px">Download email draft</a><span id="copy-status" style="font-size:12px;color:#5a6470">Paste into an Outlook email.</span></div>'
    script = '''<script>
async function copyEmail(){
 const box=document.getElementById('email-content');
 const status=document.getElementById('copy-status');
 try {
   if(navigator.clipboard && window.ClipboardItem){
     await navigator.clipboard.write([new ClipboardItem({'text/html':new Blob([box.outerHTML],{type:'text/html'}),'text/plain':new Blob([box.innerText],{type:'text/plain'})})]);
   } else {
     const selection=window.getSelection(); const range=document.createRange(); range.selectNodeContents(box);
     selection.removeAllRanges(); selection.addRange(range);
     if(!document.execCommand('copy')) throw new Error('Clipboard unavailable');
     selection.removeAllRanges();
   }
   status.textContent='Copied. Paste into your email compose window.';
 } catch(error) {
   const selection=window.getSelection(); const range=document.createRange(); range.selectNodeContents(box);
   selection.removeAllRanges(); selection.addRange(range);
   status.textContent='Email selected. Press Ctrl+C (Windows) or Command+C (Mac), then paste.';
 }
}
</script>'''
    return document.replace('<div id="email-content"', toolbar + '<div id="email-content"', 1).replace('</body>', script + '</body>')


def build_report(client: Client, state: State, day: date, mode: str,
                 include_gem: bool, include_expired: bool, lookback: int) -> dict:
    # Independent host groups may run in parallel, but do not fetch multiple
    # etnet pages at the same time through the same corporate proxy.
    months = month_keys(day)

    def get_etnet_months():
        return {month: client.text(LOCKUP_URL.format(month=month)) for month in months}

    with ThreadPoolExecutor(max_workers=2, thread_name_prefix='hk_equities') as pool:
        phip_future = pool.submit(fetch_phip, client, day, include_gem)
        etnet_future = pool.submit(get_etnet_months)
        ipos = parse_ipos(client.text(MAIN_BOARD_URL))
        trading_dates = recent_trading_dates(client.text(HOLIDAYS_URL), day)
        snapshots = fetch_short_history(client, day, trading_dates[0], lookback=lookback)
        short = snapshots[-1]
        changes = short_history_changes(snapshots)
        phip = phip_future.result()  # Raises if any PHIP source fails verification.
        etnet_pages = etnet_future.result()  # Rejects partial HTTP responses.
        lockups: dict[tuple, Lockup] = {}
        for month in months:
            for row in parse_lockups(etnet_pages[month], month):
                if include_expired or parse_date(row.expiry) >= day:
                    lockups[(row.code, row.expiry, row.holder_type)] = row

    lockup_rows = [asdict(r) for r in sorted(lockups.values(), key=lambda r: (r.expiry, r.code, r.holder_type))]
    bloomberg_audit = enrich_lockup_rows(lockup_rows)
    baseline = {"effective": snapshots[0].effective, "url": snapshots[0].url}
    added = {event["code"]: short.securities[event["code"]] for event in changes
             if event["change"] == "ADDED" and event["code"] in short.securities}
    removed = {event["code"]: event["name"] for event in changes
               if event["change"] == "REMOVED" and event["code"] not in short.securities}
    previous = state.previous(day, mode)
    notes: list[str] = []
    if previous:
        for filing in phip:
            filing.new = filing.key not in previous["phip"]
    if parse_date(short.effective) < day:
        notes.append(f'Latest available HKEX shortable list is effective {short.effective}; no later dated page was found through {day}.')
    notes.append("Lock-up window includes the current and next calendar month" +
                 (", including earlier expiries this month." if include_expired else ", with expiries on or after the report date."))
    notes.append("The IPO source lists current announcements and prospectuses; it does not provide scheduled listing dates.")
    return {"day": day.isoformat(), "generated_at": datetime.now(HK).isoformat(), "mode": mode,
            "include_expired": include_expired,
            "months": months, "phip_start": (day - timedelta(days=6)).isoformat(),
            "ipos": [asdict(r) for r in ipos], "phip": [asdict(r) for r in phip],
            "short": asdict(short), "baseline": baseline, "added": added, "removed": removed,
            "short_changes": changes,
            "short_window": {"days": 5, "start": trading_dates[0].isoformat(), "end": day.isoformat(),
                             "label": "past 5 trading days", "dates": [d.isoformat() for d in trading_dates],
                             "lists": [{"effective": item.effective, "url": item.url,
                                        "count": len(item.securities)} for item in snapshots]},
            "lockups": lockup_rows, "bloomberg": bloomberg_audit,
            "notes": notes}


def atomic_write(path: Path, data: bytes) -> None:
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_bytes(data)
    temporary.replace(path)




def generate_draft(directory: Path, day: date | None = None):
    """Collect and validate sources, then write the Outlook email and audit files."""
    day = day or datetime.now(HK).date()
    directory = Path(directory).expanduser().resolve()
    directory.mkdir(parents=True, exist_ok=True)
    with run_lock(directory / '.history'):
        state = State(directory / '.history' / 'daily.sqlite3')
        try:
            client = Client()
            report = build_report(client, state, day, 'draft', INCLUDE_GEM_PHIPS,
                                  INCLUDE_EXPIRED_THIS_MONTH, SHORT_HISTORY_LOOKBACK_DAYS)
            report['manual_sources'] = dict(client.manual_sources)
            report['cached_sources'] = dict(client.cached_sources)
            report['source_asof'] = {}
            for month in report['months']:
                source = client.cache[LOCKUP_URL.format(month=month)]
                match = re.search(r'解禁總金額\s*=.*?更新[：:]\s*(\d{1,2}/\d{1,2}/\d{4}\s+\d{1,2}:\d{2})', source, re.S)
                if not match:
                    if any(row['expiry'].replace('-', '').startswith(month) for row in report['lockups']):
                        raise SourceError(f'etnet quote timestamp missing for {month}; draft withheld.')
                    report['source_asof'][month] = None
                    continue
                quote_time = datetime.strptime(match[1], '%d/%m/%Y %H:%M').replace(tzinfo=HK)
                if quote_time.date() > datetime.now(HK).date():
                    raise SourceError(f'Future etnet quote timestamp: {match[1]}')
                report['source_asof'][month] = match[1]

            sections = annotate_section_changes(report, state.previous_sections(day, 'draft'))
            email_html, plain_text = render_report(report)
            message = make_email(report, email_html, plain_text)
            base = directory / f'HK_Equities_{day.isoformat()}'
            raw_email_path = directory / (base.name + '_email.html')
            preview_path = directory / (base.name + '_preview.html')
            eml_path = base.with_suffix('.eml')
            files = {
                raw_email_path: email_html.encode('utf-8'),
                preview_path: preview_document(email_html, eml_path.name).encode('utf-8'),
                eml_path: message.as_bytes(),
                base.with_suffix('.txt'): plain_text.encode('utf-8'),
                base.with_suffix('.json'): json.dumps(report, ensure_ascii=False, indent=2).encode('utf-8'),
            }
            for target, contents in files.items():
                atomic_write(target, contents)
            state.save(day, 'draft', ShortList(**report['short']),
                       [PHIP(**item) for item in report['phip']], None,
                       sections=sections, generated_at=report['generated_at'])
            return preview_path, raw_email_path, eml_path, report
        finally:
            state.close()



def report_summary(report):
    review = sum(bool(r['issue']) for r in report['lockups'])
    added = sum(event['change'] == 'ADDED' for event in report['short_changes'])
    removed = sum(event['change'] == 'REMOVED' for event in report['short_changes'])
    return (f'{len(report["ipos"])} IPOs · {len(report["phip"])} PHIPs · '
            f'+{added}/−{removed} short-list changes over 5 days\n'
            f'{len(report["lockups"])} lock-up tranches ({review} requiring review)')




def _ps_quote(value: str) -> str:
    """Safely embed a path or subject as a literal in PowerShell source."""
    return "'" + str(value).replace("'", "''") + "'"


def _prepend_to_outlook_signature(report_html: str, existing_signature_html: str) -> str:
    """Insert the digest above Outlook's configured personal/corporate signature."""
    signature = existing_signature_html or ''
    if not signature.strip():
        return report_html
    match = re.search(r'<body[^>]*>(.*?)</body>', report_html, re.I | re.S)
    content = match.group(1) if match else report_html
    body_start = re.search(r'<body[^>]*>', signature, re.I | re.S)
    if body_start:
        return signature[:body_start.end()] + content + '<div style="height:18px"></div>' + signature[body_start.end():]
    return report_html + '<div style="height:18px"></div>' + signature


def open_classic_outlook_draft(raw_email_path: Path, report: dict) -> tuple[bool, str]:
    """Create a saved, editable Outlook draft, opening it without sending.

    Classic Outlook for Windows supports COM. Try pywin32 first; if it is not
    installed or COM fails, use Windows PowerShell COM without installing a package.
    New Outlook / web Outlook generally do not support local COM automation.
    """
    if os.name != 'nt':
        return False, 'This is not Windows; local Classic Outlook automation was skipped.'

    subject = email_subject(report)
    html_body = raw_email_path.read_text(encoding='utf-8')
    errors = []
    try:
        import win32com.client  # optional; no pip install necessary when PowerShell works
        outlook = win32com.client.Dispatch('Outlook.Application')
        draft = outlook.CreateItem(0)
        draft.Subject = subject
        draft.To = EMAIL_TO.replace(',', ';').strip()
        draft.CC = EMAIL_CC.replace(',', ';').strip()
        draft.Display()   # Let Outlook insert the user's configured signature.
        signature_html = draft.HTMLBody
        draft.HTMLBody = _prepend_to_outlook_signature(html_body, signature_html)
        draft.Save()      # Save to the default Outlook account's Drafts folder; NO Send call.
        return True, 'Outlook editable draft is open and has been saved to Drafts.'
    except Exception as exc:
        errors.append('pywin32: ' + str(exc))

    # PowerShell is a useful fallback on managed desktops without pywin32.
    # All user text and paths are represented as quoted literals, never shell args.
    commands = '\n'.join([
        "$ErrorActionPreference = 'Stop'",
        "$outlook = New-Object -ComObject Outlook.Application",
        "$mail = $outlook.CreateItem(0)",
        '$mail.Subject = ' + _ps_quote(subject),
        '$mail.To = ' + _ps_quote(EMAIL_TO.replace(',', ';')),
        '$mail.CC = ' + _ps_quote(EMAIL_CC.replace(',', ';')),
        '$mail.Display()',
        '$signature = $mail.HTMLBody',
        '$html = [System.IO.File]::ReadAllText(' + _ps_quote(str(raw_email_path)) + ', [System.Text.Encoding]::UTF8)',
        "$matchedBody = [regex]::Match($html, '(?is)<body[^>]*>(.*?)</body>')",
        '$content = if ($matchedBody.Success) { $matchedBody.Groups[1].Value } else { $html }',
        "$start = [regex]::Match($signature, '(?is)<body[^>]*>')",
        'if ($start.Success) { $mail.HTMLBody = $signature.Insert($start.Index + $start.Length, $content + \'<div style="height:18px"></div>\') }',
        'else { $mail.HTMLBody = $html + $signature }',
        '$mail.Save()',
    ])
    encoded = base64.b64encode(commands.encode('utf-16le')).decode('ascii')
    try:
        result = subprocess.run(
            ['powershell.exe', '-NoProfile', '-NonInteractive', '-EncodedCommand', encoded],
            capture_output=True, text=True, timeout=35, check=False,
        )
        if result.returncode == 0:
            return True, 'Outlook editable draft opened using built-in Windows PowerShell.'
        errors.append('PowerShell: ' + (result.stderr or result.stdout or 'unknown error').strip()[:400])
    except (OSError, subprocess.TimeoutExpired) as exc:
        errors.append('PowerShell: ' + str(exc))
    return False, 'Classic Outlook automation unavailable (New Outlook or IT restrictions possible). ' + ' | '.join(errors)


def run_now():
    """Call exactly once when the cell is run; no scheduler or automatic sending."""
    output = (ROOT / OUTPUT_FOLDER).expanduser().resolve()
    print('HK EQUITIES — OUTLOOK CLEAR VERSION (same proven RESILIENT network)', flush=True)
    print('Checking live public sources: HKEX, Hong Kong holiday calendar, etnet ...', flush=True)
    print('PHIP and etnet fetch in parallel; etnet months run one at a time.', flush=True)
    print('Verified same-day etnet cache: ON (max ' + str(ETNET_CACHE_MAX_MINUTES) + ' minutes).')
    if HTTPS_PROXY_URL.strip():
        print('Using configured corporate proxy: ' + urlparse(HTTPS_PROXY_URL).netloc)
    print('Email will NOT be sent automatically.\n')
    try:
        preview_path, raw_email_path, eml_path, report = generate_draft(output)
    except SourceNetworkError as exc:
        print('\nSOURCE DOWNLOAD FAILED — DRAFT NOT CREATED')
        print(str(exc))
        print('\nThis can be a stalled website response or a corporate proxy restriction;')
        print('a working proxy alone does not guarantee every destination works.')
        month_match = re.search(r'ipo-free-float\.php\?month=(\d{6})', str(exc))
        if month_match and ALLOW_MANUAL_ETNET_HTML:
            month = month_match.group(1)
            folder = (ROOT / MANUAL_ETNET_FOLDER).resolve()
            print('If the page opens in Edge, save it as HTML (full page) here:')
            print(str(folder / ('etnet_' + month + '.html')))
            print('Then rerun this notebook. The saved page must be complete; source validation is mandatory.')
            print('Already successful monthly etnet downloads are retained in same-day verified cache.')
        print('If the same page fails in Edge, check with IT for permitted access.')
        print('No incomplete report was generated; no email was sent.')
        return None
    except SourceError as exc:
        print('\nSOURCE COULD NOT BE VERIFIED — DRAFT NOT CREATED')
        print(type(exc).__name__ + ': ' + str(exc))
        print('No incomplete report was generated; no email was sent.')
        return None

    print('\nDRAFT READY: ' + email_subject(report))
    print(report_summary(report))
    print('Files saved in:', output)
    print('Editable email file:', eml_path.name)
    print('Preview:', preview_path.name)
    print('JSON audit trail:', report['day'] + ' / ' + str(len(report['short_changes'])) + ' short-list events')
    if report.get('manual_sources'):
        print('SOURCE REVIEW: manually saved etnet HTML used. Confirm timestamps before distribution.')
    if report.get('cached_sources'):
        print('SOURCE REVIEW: etnet source reused from verified same-day cache:', report['cached_sources'])
    if any(row['issue'] for row in report['lockups']):
        print('DATA REVIEW: some lock-up records were flagged in amber (retained, not silently dropped).')
    if report.get('bloomberg', {}).get('tranches_with_unavailable_ratios'):
        print('BLOOMBERG REVIEW: unavailable ratios are N/A; reasons are shown in the draft and JSON audit.')
        if report['bloomberg'].get('error'):
            print(report['bloomberg']['error'])

    launched, detail = (False, 'Outlook opening disabled in settings.')
    if OPEN_OUTLOOK:
        launched, detail = open_classic_outlook_draft(raw_email_path, report)
    print('\n' + detail)

    if not launched and SHOW_NOTEBOOK_PREVIEW_IF_NO_OUTLOOK:
        print('Showing the formatted preview within Jupyter instead. Use the .eml file or copy it into Outlook.')
        try:
            from IPython.display import HTML, FileLink, display
            display(FileLink(str(eml_path.relative_to(ROOT))))
            display(HTML(raw_email_path.read_text(encoding='utf-8')))
        except ImportError:
            print('View the HTML preview in your browser:', preview_path)
            webbrowser.open(preview_path.as_uri())
    return {'preview': preview_path, 'email_html': raw_email_path, 'eml': eml_path, 'report': report}


# This guard is True in a pasted Jupyter code cell or when run as a .py script.
# Importing as a library will not accidentally generate drafts.
if __name__ == '__main__':
    LAST_DRAFT = run_now()

In [ ]:
# Paste this entire file into a NEW second Jupyter cell, below your existing code.
# Run the original cell first. It must finish successfully and create LAST_DRAFT.
# The original code, LAST_DRAFT, and its first email are not modified.
# Bloomberg uses your existing Desktop API connection. LAST_PRICE corresponds
# to =BDP(ticker,"LAST_PRICE"); historical closes use BDH's PX_LAST field.
# IPO price uses =BDP(ticker,"EQY_INIT_PO_SH_PX").

LOCKUP_EMAIL_TO = globals().get("EMAIL_TO", "")
LOCKUP_EMAIL_CC = globals().get("EMAIL_CC", "")
LOCKUP_EMAIL_OPEN_OUTLOOK = globals().get("OPEN_OUTLOOK", True)
LOCKUP_EMAIL_CHECK_A_SHARES = True  # etnet A/H pairs + Eastmoney share classes
LOCKUP_EMAIL_WEB_TIMEOUT = 8        # Per network operation, seconds
LOCKUP_EMAIL_WEB_BUDGET = 45        # Stop starting/reading requests after this budget


def _lu_select(report):
    """Reuse the first email's actual month window; include YES and NEW."""
    import calendar
    import copy
    from datetime import date

    start = date.fromisoformat(report["day"])
    months = set(report["months"])
    last_month = max(months)
    year, month = int(last_month[:4]), int(last_month[4:])
    end = date(year, month, calendar.monthrange(year, month)[1])
    eligible = report["short"]["securities"]
    rows = []
    for original in report["lockups"]:
        expiry = date.fromisoformat(original["expiry"])
        if (start <= expiry <= end and expiry.strftime("%Y%m") in months
                and original["code"] in eligible):
            row = copy.deepcopy(original)
            row["shortable"] = "NEW" if row["code"] in report.get("added", {}) else "YES"
            rows.append(row)
    return sorted(rows, key=lambda r: (r["expiry"], r["code"], r["holder_type"])), start, end


def _lu_a_share_profile(document, code, parse_html):
    """Infer mainland A-share presence from a validated issuer capital breakdown."""
    import re

    root = parse_html(document)
    title = root.find('title')
    if (not title or not re.match(r'^0*' + str(int(code)) + r'\.HK\b', title.get_text(strip=True))
            or not re.search(r'</html\s*>', document, re.I)):
        raise ValueError('Incomplete profile or stock identity mismatch.')
    sections = []
    for tr in root.find_all('tr'):
        cells = tr.find_all('td', recursive=False)
        if len(cells) == 2 and cells[0].get_text(strip=True) == '股本':
            sections.append(cells[1])
    if len(sections) != 1:
        raise ValueError('Missing or ambiguous share-capital section.')
    amounts = {}
    for tr in sections[0].find_all('tr'):
        cells = tr.find_all('td', recursive=False)
        if len(cells) < 2:
            raise ValueError('Incomplete share-capital row.')
        label, value = cells[0].get_text(strip=True), cells[1].get_text(strip=True)
        if label in amounts or not re.fullmatch(r'(?:\d+|\d{1,3}(?:,\d{3})+)', value):
            raise ValueError('Invalid or duplicate share-capital value.')
        amounts[label] = int(value.replace(',', ''))
    total = amounts.get('發行股數', 0)
    classes = {k: v for k, v in amounts.items() if k != '發行股數'}
    mainland_a = {'有限制流通A股', '無限制流通A股', 'A股'}
    non_a = {'H股', '內資股及其他', '內資股', '非上市外資股', '普通股'}
    # An issued total alone is not evidence of No. Unlisted domestic shares
    # and offshore Class A voting shares are not mainland-listed A shares.
    if (total <= 0 or not classes or not set(classes) <= mainland_a | non_a
            or sum(classes.values()) != total):
        raise ValueError('Share classes do not provide a complete recognised capital breakdown.')
    return {'status': 'YES' if any(classes.get(k, 0) > 0 for k in mainland_a) else 'NO',
            'capital': amounts, 'error': None,
            'method': 'etnet issuer capital classes; not a Bloomberg field or exchange confirmation'}


def _lu_a_share_roster(document, parse_html):
    """Read exact H/A ticker pairs; absence from this roster never proves No."""
    import re
    from urllib.parse import urlparse, parse_qs

    if not re.search(r'</html\s*>', document, re.I):
        raise ValueError('Incomplete A/H table page.')
    root = parse_html(document)
    tables = []
    for table in root.find_all('table'):
        rows = table.find_all('tr')
        if rows:
            heading = rows[0].get_text(' ', strip=True)
            if 'H股代號' in heading and 'A股代號' in heading:
                tables.append(rows)
    if len(tables) != 1:
        raise ValueError('Missing or ambiguous A/H table.')
    pairs = {}
    for row in tables[0][1:]:
        cells = row.find_all('td', recursive=False)
        if len(cells) != 9:
            raise ValueError('Unexpected A/H row shape.')
        h_code = cells[1].get_text(strip=True)
        a_match = re.fullmatch(r'(滬|深|京)(\d{6})', cells[5].get_text(strip=True))
        if not re.fullmatch(r'\d{5}', h_code) or not a_match:
            raise ValueError('Invalid A/H ticker pair.')
        a_venue, a_code = a_match.groups()
        for cell, path, code in ((cells[1], 'realtime/quote.php', h_code),
                                 (cells[5], 'ashares/quote.php', a_code)):
            anchor = cell.find('a')
            link = urlparse(anchor.get('href', '') if anchor else '')
            if not link.path.endswith(path) or parse_qs(link.query).get('code') != [code]:
                raise ValueError('A/H ticker does not match its source link.')
        if h_code in pairs:
            raise ValueError('Duplicate H-share ticker in A/H table.')
        pairs[h_code] = {'a_code': a_code,
                         'a_exchange': {'滬': 'Shanghai', '深': 'Shenzhen', '京': 'Beijing'}[a_venue]}
    if len(pairs) < 100:
        raise ValueError('A/H table unexpectedly small; page may be incomplete.')
    return pairs


def _lu_source_text(client, url, env, deadline):
    """Reuse the original proxy/TLS opener, with bounded reads and no retry loop."""
    import time
    import urllib.request

    remaining = deadline - time.monotonic()
    timeout = float(env.get('LOCKUP_EMAIL_WEB_TIMEOUT', 8))
    if not 0 < timeout <= 30 or remaining <= 0:
        raise TimeoutError('A-share network budget exhausted or timeout setting invalid.')
    # In-memory source adapters used by the offline checks have only text().
    if not hasattr(client, 'opener'):
        return client.text(url)
    if url in client.cache:
        return client.cache[url]
    request = urllib.request.Request(url, headers={
        'User-Agent': 'Mozilla/5.0', 'Accept': 'application/json,text/html',
        'Accept-Encoding': 'identity', 'Connection': 'close', 'Cache-Control': 'no-cache'})
    with client.opener.open(request, timeout=min(timeout, remaining)) as response:
        chunks, size = [], 0
        while True:
            if time.monotonic() >= deadline:
                raise TimeoutError('A-share network budget exhausted.')
            chunk = response.read1(64 * 1024)
            if not chunk:
                break
            size += len(chunk)
            if size > 2_000_000:
                raise ValueError('A-share response exceeds size limit.')
            chunks.append(chunk)
        raw = b''.join(chunks)
        length = response.headers.get('Content-Length')
        if length and length.isdecimal() and int(length) != size:
            raise ValueError('Incomplete A-share response.')
        raw = env['_decode_http_encoding'](raw, response.headers.get('Content-Encoding'), url, limit=2_000_000)
        return env['_decode_source'](url, raw, response.headers.get_content_charset())


def _lu_share_structure(document, codes, day):
    """Use complete, dated ordinary-share classes; mainland-listed is not automatically A."""
    import json
    from datetime import date
    from decimal import Decimal

    data = json.loads(document)
    result = data.get('result') or {}
    rows = result.get('data')
    if (data.get('success') is not True or not isinstance(rows, list)
            or result.get('pages') != 1 or result.get('count') != len(rows) or len(rows) > 500):
        raise ValueError('Incomplete or unsuccessful share-structure response.')
    groups = {code: [] for code in codes}
    for row in rows:
        code = row.get('SECURITY_CODE')
        if code not in groups or row.get('SECUCODE') != code + '.HK':
            raise ValueError('Unexpected issuer in share-structure response.')
        groups[code].append(row)
    labels = {'001': '已发行普通股', '001001': '香港普通股', '001002': '非港股',
              '001002001': '内地上市股', '001002002': '海外上市股', '001002003': '非上市流通股'}
    output = {}
    for code, entries in groups.items():
        item = {'status': 'N/A', 'error': None}
        try:
            amounts, dates = {}, set()
            for row in entries:
                key = row.get('SHARES_TYPE_CODE')
                if key not in labels or key in amounts or row.get('SHARES_TYPE') != labels[key]:
                    raise ValueError('Unknown or duplicate share class.')
                value = Decimal(str(row['SHARES_NUM']))
                if not value.is_finite() or value < 0 or value != value.to_integral_value():
                    raise ValueError('Invalid share count.')
                amounts[key] = int(value)
                dates.add(date.fromisoformat(row['CHANGE_DATE'][:10]))
            if len(dates) != 1 or next(iter(dates)) > day:
                raise ValueError('Missing, mixed or future share-structure date.')
            total, hk = amounts.get('001'), amounts.get('001001')
            non_hk = amounts.get('001002', 0)
            leaves = sum(amounts.get(k, 0) for k in ('001002001', '001002002', '001002003'))
            if (total is None or hk is None or total <= 0 or hk <= 0
                    or total != hk + non_hk or non_hk != leaves):
                raise ValueError('Share classes do not reconcile to the complete issued total.')
            item.update(status='MAINLAND_LISTED' if amounts.get('001002001', 0) > 0 else 'NO',
                        capital={labels[k]: v for k, v in amounts.items()},
                        asof=next(iter(dates)).isoformat(),
                        method='Eastmoney complete ordinary-share structure, RPT_HKF10_INFO_EQUITYSTR')
        except Exception as exc:
            item['error'] = str(exc)[:600]
        output[code] = item
    return output


def _lu_a_shares(codes, env):
    """Batch listings and share classes; profile fallback only for unresolved issuers."""
    from datetime import datetime
    from urllib.parse import urlencode
    import hashlib
    import re
    import time

    results, pairs, structures = {}, {}, {}
    keys = sorted(set(codes))
    if not keys:
        return results
    day = datetime.now(env['HK']).date()
    for code in keys:
        results[code] = {'status': 'N/A', 'error': None, 'retrieved_at': datetime.now(env['HK']).isoformat()}
    try:
        if not env.get('LOCKUP_EMAIL_CHECK_A_SHARES', True):
            raise ValueError('A-share lookup disabled.')
        budget = float(env.get('LOCKUP_EMAIL_WEB_BUDGET', 45))
        if not 0 < budget <= 120:
            raise ValueError('A-share network budget must be between 0 and 120 seconds.')
        deadline = time.monotonic() + budget
        client = env['Client']()
    except Exception as exc:
        for result in results.values():
            result['error'] = str(exc)[:600]
        return results
    valid = [c for c in keys if isinstance(c, str) and re.fullmatch(r'\d{5}', c)]
    def fetch(url):
        record = {'url': url, 'retrieved_at': datetime.now(env['HK']).isoformat()}
        document = _lu_source_text(client, url, env, deadline)
        record['document_sha256'] = hashlib.sha256(document.encode('utf-8')).hexdigest()
        return document, record
    roster = {'url': 'https://www.etnet.com.hk/www/tc/stocks/ah.php'}
    try:
        document, roster = fetch(roster['url'])
        pairs = _lu_a_share_roster(document, env['parse_html'])
        roster['count'] = len(pairs)
    except Exception as exc:
        roster['error'] = str(exc)[:600]
    for offset in range(0, len(valid), 40):
        batch = valid[offset:offset + 40]
        url = 'https://datacenter.eastmoney.com/securities/api/data/v1/get?' + urlencode({
            'reportName': 'RPT_HKF10_INFO_EQUITYSTR',
            'columns': 'SECUCODE,SECURITY_CODE,SECURITY_NAME_ABBR,CHANGE_DATE,SHARES_TYPE_CODE,SHARES_TYPE,SHARES_NUM',
            'filter': '(SECUCODE in ("' + '","'.join(c + '.HK' for c in batch) + '"))',
            'pageNumber': '1', 'pageSize': '500', 'sortTypes': '1,1',
            'sortColumns': 'SECURITY_CODE,SHARES_TYPE_CODE', 'source': 'F10', 'client': 'PC'})
        record = {'url': url}
        try:
            document, record = fetch(url)
            parsed = _lu_share_structure(document, batch, day)
            for code, item in parsed.items():
                structures[code] = dict(item, source=record)
        except Exception as exc:
            for code in batch:
                structures[code] = {'status': 'N/A', 'error': str(exc)[:600], 'source': record}
    for code, result in results.items():
        if code not in valid:
            result['error'] = 'Invalid Hong Kong stock code.'
            continue
        structure = structures[code]
        pair = pairs.get(code)
        result.update(roster=dict(roster), share_structure=structure)
        if pair and structure['status'] == 'NO':
            result.update(pair)
            result['error'] = 'Explicit A/H listing conflicts with the complete share structure.'
        elif pair:
            result.update(pair, status='YES', method='Explicit etnet A/H ticker pair')
        elif structure['status'] == 'NO':
            result.update(status='NO', method=structure['method'])
        else:
            url = 'https://www.etnet.com.hk/www/tc/stocks/realtime/quote_ci_brief.php?code=' + str(int(code))
            try:
                document, record = fetch(url)
                profile = _lu_a_share_profile(document, code, env['parse_html'])
                result['profile'] = dict(profile, source=record)
                if profile['status'] == 'NO' and structure['status'] == 'MAINLAND_LISTED':
                    # "Mainland listed" also includes NEEQ domestic shares.
                    # Use the more specific capital classes only if both sources
                    # account for exactly the same total, H and domestic shares.
                    p, s = profile['capital'], structure['capital']
                    domestic = sum(p.get(k, 0) for k in ('內資股及其他', '內資股'))
                    if (p['發行股數'] != s['已发行普通股']
                            or p.get('H股') != s['香港普通股']
                            or domestic != s.get('内地上市股')
                            or domestic + p['H股'] != p['發行股數']):
                        raise ValueError('Domestic share classes disagree between sources.')
                    profile['method'] += '; domestic/H counts reconciled to Eastmoney (mainland-listed is not necessarily A)'
                result.update(status=profile['status'], method=profile['method'])
            except Exception as exc:
                result['error'] = str(exc)[:600]
    return results


def _lu_horizons(day):
    import calendar
    from datetime import date, timedelta

    year, month = (day.year - 1, 12) if day.month == 1 else (day.year, day.month - 1)
    previous_month = date(year, month, min(day.day, calendar.monthrange(year, month)[1]))
    return {"7D": day - timedelta(days=7), "1M": previous_month}


def _lu_bbg_request(client, securities, day, historical=False, reference_fields=None):
    """Bounded, sequential Bloomberg request; discard incomplete request results."""
    import time
    from datetime import date, datetime, timedelta

    # Validate before submitting so a bad setting cannot strand a live request.
    timeout = float(globals().get("BLOOMBERG_REQUEST_TIMEOUT_SECONDS", 30))
    if not 0 < timeout <= 120:
        raise ValueError("Bloomberg request timeout must be between 0 and 120 seconds.")
    fields = ["PX_LAST"] if historical else (reference_fields or ["LAST_PRICE", "CRNCY", "LAST_UPDATE_DT", "EQY_INIT_PO_SH_PX"])
    request = client.service.createRequest("HistoricalDataRequest" if historical else "ReferenceDataRequest")
    for security in securities:
        request.getElement("securities").appendValue(security)
    for field in fields:
        request.getElement("fields").appendValue(field)
    if historical:
        # Observed daily closes only. Do not synthesize holiday observations.
        # Cash dividends excluded; Bloomberg capital-change adjustments enabled.
        start = _lu_horizons(day)["1M"] - timedelta(days=14)
        for key, value in {
            "startDate": start.strftime("%Y%m%d"), "endDate": day.strftime("%Y%m%d"),
            "periodicitySelection": "DAILY", "periodicityAdjustment": "ACTUAL",
            "nonTradingDayFillOption": "ACTIVE_DAYS_ONLY", "nonTradingDayFillMethod": "NIL_VALUE",
            "adjustmentFollowDPDF": False, "adjustmentNormal": False,
            "adjustmentAbnormal": False, "adjustmentSplit": True,
            "pricingOption": "PRICING_OPTION_PRICE", "overrideOption": "OVERRIDE_OPTION_CLOSE",
        }.items():
            request.set(key, value)
    result = {s: {"fields": {}, "history": {}, "errors": {}} for s in securities}
    queue = client.api.EventQueue()
    correlation = client.session.sendRequest(request, eventQueue=queue)
    deadline, complete = time.monotonic() + timeout, False
    seen = set()
    try:
        while True:
            remaining = deadline - time.monotonic()
            if remaining <= 0:
                raise RuntimeError("Bloomberg request timed out; incomplete response discarded.")
            event = queue.nextEvent(max(1, min(500, int(remaining * 1000))))
            kind = event.eventType()
            for message in event:
                if kind == client.api.Event.REQUEST_STATUS:
                    raise RuntimeError(str(message)[:600])
                if kind not in (client.api.Event.PARTIAL_RESPONSE, client.api.Event.RESPONSE):
                    continue
                if message.hasElement("responseError"):
                    raise RuntimeError(str(message.getElement("responseError"))[:600])
                if not message.hasElement("securityData"):
                    raise RuntimeError("Bloomberg response has no securityData.")
                data = message.getElement("securityData")
                items = [data] if historical else [data.getValueAsElement(i) for i in range(data.numValues())]
                for item in items:
                    security = item.getElementAsString("security")
                    if item.hasElement("sequenceNumber"):
                        sequence = item.getElementAsInteger("sequenceNumber")
                        if not 0 <= sequence < len(securities):
                            raise RuntimeError("Invalid Bloomberg security sequence number.")
                        security = securities[sequence]
                    if security not in result:
                        raise RuntimeError("Unexpected Bloomberg security: " + security)
                    seen.add(security)
                    entry = result[security]
                    if item.hasElement("securityError"):
                        entry["errors"]["security"] = str(item.getElement("securityError"))[:600]
                        entry["fields"].clear()
                        entry["history"].clear()
                        continue
                    if item.hasElement("fieldExceptions"):
                        errors = item.getElement("fieldExceptions")
                        for i in range(errors.numValues()):
                            error = errors.getValueAsElement(i)
                            entry["errors"][error.getElementAsString("fieldId")] = str(error.getElement("errorInfo"))[:600]
                    if not item.hasElement("fieldData"):
                        continue
                    values = item.getElement("fieldData")
                    if historical:
                        if "PX_LAST" in entry["errors"]:
                            entry["history"].clear()
                            continue
                        for i in range(values.numValues()):
                            point = values.getValueAsElement(i)
                            if not point.hasElement("date") or not point.hasElement("PX_LAST"):
                                continue
                            price = point.getElement("PX_LAST")
                            if price.isNull():
                                continue
                            observed = point.getElementAsDatetime("date")
                            if isinstance(observed, datetime):
                                observed = observed.date()
                            if not isinstance(observed, date) or not start <= observed <= day:
                                raise RuntimeError("Invalid Bloomberg historical observation date.")
                            key, value = observed.isoformat(), price.getValueAsString()
                            if key in entry["history"] and entry["history"][key] != value:
                                raise RuntimeError("Conflicting Bloomberg historical observations.")
                            entry["history"][key] = value
                    else:
                        for field in fields:
                            if field in entry["errors"] or not values.hasElement(field):
                                continue
                            element = values.getElement(field)
                            if element.isNull() or not element.numValues():
                                continue
                            if field == "LAST_UPDATE_DT":
                                try:
                                    observed = element.getValueAsDatetime()
                                    if isinstance(observed, datetime):
                                        observed = observed.date()
                                    if not isinstance(observed, date):
                                        raise ValueError("Quote date is not a date.")
                                    entry["fields"][field] = observed.isoformat()
                                except Exception:
                                    entry["errors"][field] = "Quote date unavailable or invalid."
                            else:
                                entry["fields"][field] = element.getValueAsString()
            if kind == client.api.Event.RESPONSE:
                complete = True
                break
        for security in set(securities) - seen:
            result[security]["errors"]["security"] = "Security omitted from Bloomberg response."
        return result
    finally:
        if not complete:
            try:
                client.session.cancel(correlation)
            except Exception:
                pass


def _lu_number(value):
    from decimal import Decimal, InvalidOperation

    if value is None or isinstance(value, bool):
        return None
    try:
        number = Decimal(str(value))
        return number if number.is_finite() else None
    except (InvalidOperation, ValueError):
        return None


def _lu_performance(snapshot, history, day):
    """Price returns from last available close on/before each calendar target."""
    from datetime import date

    fields = snapshot.get("fields", {})
    errors = dict(snapshot.get("errors", {}))
    errors.update({"history/" + k: v for k, v in history.get("errors", {}).items()})
    price = _lu_number(fields.get("LAST_PRICE"))
    if any(k in snapshot.get("errors", {}) for k in ("security", "request", "LAST_PRICE")):
        price = None
    if price is None or price <= 0:
        price = None
        errors.setdefault("LAST_PRICE", "Last price unavailable.")
    quote_date = fields.get("LAST_UPDATE_DT")
    if quote_date:
        try:
            if date.fromisoformat(quote_date) > day:
                price = None
                errors["LAST_PRICE"] = "Future-dated quote withheld."
        except ValueError:
            quote_date = None
    if not quote_date:
        errors.setdefault("LAST_UPDATE_DT", "Quote date unavailable.")
    if not fields.get("CRNCY"):
        errors.setdefault("CRNCY", "Price currency unavailable.")
    current = bool(quote_date and 0 <= (day - date.fromisoformat(quote_date)).days <= 10)
    ipo_price = _lu_number(fields.get("EQY_INIT_PO_SH_PX"))
    if any(k in snapshot.get("errors", {}) for k in ("security", "request", "EQY_INIT_PO_SH_PX")):
        ipo_price = None
    if ipo_price is None or ipo_price <= 0:
        ipo_price = None
        errors.setdefault("EQY_INIT_PO_SH_PX", "IPO offer price unavailable.")
    ipo_change = (price / ipo_price - 1) * 100 if price is not None and ipo_price is not None and current else None
    if ipo_change is None:
        errors["vs_IPO"] = "IPO comparison unavailable: missing IPO/current price or missing/stale quote date."
    observations = []
    if not history.get("errors"):
        for observed, raw in history.get("history", {}).items():
            value = _lu_number(raw)
            if value is not None and value > 0:
                observations.append((date.fromisoformat(observed), value))
    out = {"last_price": str(price) if price is not None else None,
           "ipo_price": str(ipo_price) if ipo_price is not None else None,
           "ipo_change_pct": str(ipo_change) if ipo_change is not None else None,
           "currency": fields.get("CRNCY"), "quote_date": quote_date, "errors": errors}
    for label, target in _lu_horizons(day).items():
        choices = [(d, p) for d, p in observations if d <= target]
        baseline = max(choices, default=None)
        # Long suspension / very old baseline: do not label it a 7D/1M move.
        valid = baseline is not None and (target - baseline[0]).days <= 10
        chronological = baseline is not None and quote_date and baseline[0] < date.fromisoformat(quote_date)
        change = (price / baseline[1] - 1) * 100 if price is not None and valid and current and chronological else None
        out[label] = {"target_date": target.isoformat(),
                      "baseline_date": baseline[0].isoformat() if baseline else None,
                      "baseline_price": str(baseline[1]) if baseline else None,
                      "change_pct": str(change) if change is not None else None}
        if change is None:
            errors[label] = "Price change unavailable: missing, stale or insufficient price history/quote date."
    return out


def _lu_fx_rate(snapshot, day):
    """USDHKD is HKD per USD; only a positive, dated Bloomberg quote is usable."""
    from datetime import date

    fields = snapshot.get('fields', {})
    errors = dict(snapshot.get('errors', {}))
    rate = _lu_number(fields.get('LAST_PRICE'))
    observed = fields.get('LAST_UPDATE_DT')
    if rate is None or rate <= 0:
        errors.setdefault('LAST_PRICE', 'USD/HKD rate unavailable or invalid.')
    try:
        quote_date = date.fromisoformat(observed)
        if not 0 <= (day - quote_date).days <= 10:
            raise ValueError('Stale or future FX quote.')
    except (TypeError, ValueError):
        errors.setdefault('LAST_UPDATE_DT', 'USD/HKD quote date missing, stale or invalid.')
    return {'security': 'USDHKD Curncy', 'hkd_per_usd': str(rate) if not errors else None,
            'quote_date': observed, 'errors': errors}


def _lu_turnover_usd(value, fx):
    amount, rate = _lu_number(value), _lu_number(fx.get('hkd_per_usd'))
    if amount is None or amount < 0 or rate is None or rate <= 0 or fx.get('errors'):
        return None
    return str(amount / rate)


def _lu_render(report, rows, start, end, retrieved_at):
    from html import escape
    from urllib.parse import urlsplit

    def text(value):
        return escape(str(value if value is not None else "N/A"))

    def number(value, suffix="", divisor=1, signed=False, decimals=2):
        n = _lu_number(value)
        if n is None:
            return "N/A"
        n /= divisor
        return (f"{n:+,.{decimals}f}" if signed else f"{n:,.{decimals}f}") + suffix

    def change(value):
        n = _lu_number(value)
        colour = "#087544" if n is not None and n > 0 else "#b11d2b" if n is not None and n < 0 else "#555555"
        return '<b style="color:' + colour + '">' + number(value, "%", signed=True) + '</b>'

    font = "font-family:'Times New Roman',Times,serif;"

    def stock_link(row):
        # Cell 1 links the stock label to row['url']; keep that exact destination.
        label, url = text(row['code'] + ' ' + row['name']), row.get('url')
        try:
            parsed = urlsplit(url or '')
            valid = parsed.scheme.lower() in ('http', 'https') and bool(parsed.netloc)
        except ValueError:
            valid = False
        if not valid:
            return '<b>' + label + '</b>'
        return '<a href="' + text(url) + '" style="' + font + 'color:#0759A4;font-weight:bold;text-decoration:underline">' + label + '</a>'

    subject = "Shortable IPO Lock-up Expiries | " + start.strftime("%d %b %Y")
    # The email body starts with the table. Its subject stays on the email itself.
    parts, plain = [], []
    if rows:
        headers = ["Expiry", "Stock / holders", "A-share", "Unlock", "Avg 20D turnover (USDm)", "Liquidity ratios", "IPO price", "Last price", "vs IPO %", "7D %", "1M %"]
        parts.append('<table width="100%" cellspacing="0" cellpadding="6" style="' + font + 'border-collapse:collapse;font-size:14px;border-top:2px solid #CB2125"><tr>'
                     + ''.join('<th align="left" style="background:#eeeeee;border-bottom:1px solid #cccccc">'
                               + (h.replace('turnover (USDm)', '<br>turnover (USDm)') if h.startswith('Avg 20D') else h)
                               + '</th>' for h in headers) + '</tr>')
        for i, row in enumerate(rows):
            p = row["price_performance"]
            expiry = row["expiry"]
            from datetime import date
            days = (date.fromisoformat(expiry) - start).days
            holder_count = str(row["holders"])
            holders = row["holder_type"] + " · " + holder_count + (" holder" if holder_count == "1" else " holders")
            amount = ("HKDm " + number(row.get("unlock_amount_hkd"), divisor=1000000)
                      if row.get("unlock_amount_hkd") is not None else row.get("amount", "N/A"))
            quote_time = report.get("source_asof", {}).get(expiry.replace("-", "")[:6])
            unlock = '<b>' + text(row["percent"]) + '</b> of issued shares<br>' + text(amount)
            if quote_time:
                unlock += '<br><span style="font-size:11px;color:#666666">Value as of ' + text(quote_time) + ' HKT</span>'
            ratios = (number(row.get("unlock_to_float_market_cap_pct"), "%") + ' / float cap<br>'
                      + number(row.get("unlock_to_20d_avg_turnover_x"), "x") + ' / 20D turnover')
            # Convert cell 1's HKD denominator with one shared current USD/HKD quote.
            turnover = number(row.get("avg_turnover_20d_usd"), divisor=1000000)
            turnover_end = report.get("bloomberg", {}).get("turnover_end_date")
            if turnover_end:
                turnover += '<br><span style="font-size:11px;color:#666666">To ' + text(turnover_end) + '</span>'
            a_share = text(row.get("a_share", {}).get("status", "N/A"))
            last = number(p["last_price"], decimals=3) + ' ' + text(p["currency"])
            last += '<br><span style="font-size:11px;color:#666666">Quote: ' + text(p["quote_date"]) + '</span>'
            ipo = number(p["ipo_price"], decimals=3) + ' ' + text(p["currency"])
            needs_review = bool(row.get("issue") or row.get("source_issue") or row.get("bloomberg_issues") or p["errors"]
                                or row.get("a_share", {}).get("error") or row.get("avg_turnover_20d_usd") is None)
            stock = stock_link(row) + '<br>' + text(holders)
            if needs_review:
                stock += '<br><span style="font-size:11px;color:#795b00">REVIEW</span>'
            cells = ['<span style="white-space:nowrap">' + text(expiry) + '</span><br><span style="color:#666666">' + str(days) + ' days</span>',
                     stock,
                     a_share, unlock, turnover, ratios, ipo, last, change(p["ipo_change_pct"]),
                     change(p["7D"]["change_pct"]), change(p["1M"]["change_pct"])]
            percent = str(row.get('percent', '')).strip()
            unlock_pct = _lu_number(percent[:-1].strip()) if percent.endswith('%') else None
            highlights = {}
            if unlock_pct is not None and unlock_pct > 30:
                highlights[3] = ('#FFF2CC', 'color:#6B4F00;')
            if row.get('a_share', {}).get('status') == 'YES':
                highlights[2] = ('#E2F0D9', 'color:#245B28;font-weight:bold;')
            background = '#ffffff' if i % 2 == 0 else '#f7f7f7'
            formatted = []
            for index, cell in enumerate(cells):
                colour, extra = highlights.get(index, (background, ''))
                formatted.append('<td valign="top" bgcolor="' + colour + '" style="' + font + 'background-color:' + colour
                                 + ';border-bottom:1px solid #dddddd;' + extra + '">' + cell + '</td>')
            parts.append('<tr>' + ''.join(formatted) + '</tr>')
            # Detailed review reasons remain in the JSON audit, not email footnotes.
            plain.append(f"{expiry} | {row['code']} {row['name']} | {holders} | A-share {row.get('a_share', {}).get('status', 'N/A')} | Unlock {row['percent']} of issued shares / {amount} | "
                         f"Avg 20D turnover USDm {number(row.get('avg_turnover_20d_usd'), divisor=1000000)}"
                         + (f" (to {turnover_end})" if turnover_end else "") + " | "
                         f"Unlock/float cap {number(row.get('unlock_to_float_market_cap_pct'), '%')} | "
                         f"Unlock/20D turnover {number(row.get('unlock_to_20d_avg_turnover_x'), 'x')} | "
                         f"IPO price {number(p['ipo_price'], decimals=3)} {p['currency'] or 'N/A'} | "
                         f"Last {number(p['last_price'], decimals=3)} {p['currency'] or 'N/A'} (quote {p['quote_date'] or 'N/A'}) | "
                         f"vs IPO {number(p['ipo_change_pct'], '%', signed=True)} | "
                         f"7D {number(p['7D']['change_pct'], '%', signed=True)} | 1M {number(p['1M']['change_pct'], '%', signed=True)}"
                         + (f" | {row['url']}" if row.get('url') else '')
                         + (" | REVIEW" if needs_review else ""))
        parts.append('</table>')
    else:
        parts.append('<p>No shortable lock-up expiries in this window.</p>')
        plain.append('No shortable lock-up expiries in this window.')
    # Methods, source metadata and detailed review notes stay outside the email.
    body = '<!doctype html><html><head><meta charset="utf-8"></head><body style="' + font + 'color:#222222"><div id="email-content" style="' + font + 'max-width:1320px;margin:18px auto">' + ''.join(parts) + '</div></body></html>'
    return subject, body, '\n'.join(plain)


def _lu_outlook(path, subject, to, cc, env):
    import os
    import subprocess

    if os.name != "nt":
        return False, "Open the saved .eml file in Outlook; automatic draft opening requires Classic Outlook for Windows."
    body = path.read_text(encoding="utf-8")
    draft = None
    try:
        import win32com.client
        draft = win32com.client.Dispatch("Outlook.Application").CreateItem(0)
    except Exception:
        # Missing/broken pywin32 or failure before a draft exists: use PowerShell.
        pass
    if draft is not None:
        try:
            draft.Subject, draft.To, draft.CC = subject, to.replace(',', ';'), cc.replace(',', ';')
            draft.Display()
            draft.HTMLBody = env["_prepend_to_outlook_signature"](body, draft.HTMLBody)
            draft.Save()
            return True, "Second Outlook draft opened and saved. Nothing sent."
        except Exception as exc:
            # Do not create a duplicate via fallback after a potentially successful Display.
            return False, "Outlook automation did not finish; check open drafts before opening the .eml. " + str(exc)[:200]
    q = lambda s: "'" + str(s).replace("'", "''") + "'"
    script = '\n'.join([
        "$ErrorActionPreference = 'Stop'",
        "$outlook = New-Object -ComObject Outlook.Application", "$mail = $outlook.CreateItem(0)",
        "$mail.Subject = " + q(subject), "$mail.To = " + q(to.replace(',', ';')), "$mail.CC = " + q(cc.replace(',', ';')),
        "$mail.Display()", "$signature = $mail.HTMLBody",
        "$html = [System.IO.File]::ReadAllText(" + q(path) + ", [System.Text.Encoding]::UTF8)",
        "$body = [regex]::Match($html, '(?is)<body[^>]*>(.*)</body>').Groups[1].Value",
        "$start = [regex]::Match($signature, '(?is)<body[^>]*>')",
        "if ($start.Success) { $mail.HTMLBody = $signature.Insert($start.Index + $start.Length, $body + '<br>') }",
        "else { $mail.HTMLBody = $html + $signature }", "$mail.Save()",
    ])
    import base64
    encoded = base64.b64encode(script.encode('utf-16le')).decode('ascii')
    try:
        completed = subprocess.run(['powershell.exe', '-NoProfile', '-NonInteractive', '-EncodedCommand', encoded],
                                   capture_output=True, text=True, timeout=45)
        if completed.returncode == 0:
            return True, "Second Outlook draft opened and saved. Nothing sent."
        return False, "Outlook automation unavailable. Open the saved .eml; check for an existing draft first."
    except (OSError, subprocess.TimeoutExpired) as exc:
        return False, "Outlook automation did not finish; check open drafts before opening the .eml. " + str(exc)[:200]


def _lu_run(first, env, open_outlook=True):
    import copy
    import json
    from datetime import datetime
    from email.message import EmailMessage
    from email.policy import SMTP
    from email.utils import format_datetime, make_msgid
    from pathlib import Path

    required = ("HK", "BloombergClient", "bloomberg_security", "atomic_write")
    if not first or "report" not in first or "email_html" not in first or any(k not in env for k in required):
        raise RuntimeError("Run your original first cell successfully before this second cell.")
    report = copy.deepcopy(first["report"])
    rows, start, end = _lu_select(report)
    now = datetime.now(env["HK"])
    if start != now.date():
        raise RuntimeError("The first report is not dated today in Hong Kong. Rerun the original cell first.")
    securities = sorted({env["bloomberg_security"](r["code"]) for r in rows})
    snapshots, histories = {}, {}
    fx_snapshot = {'errors': {'request': 'Bloomberg FX not requested.'}}
    if securities:
        print(f"Second email: fetching Bloomberg prices for {len(securities)} unique stocks ...", flush=True)
        try:
            if not env.get("BLOOMBERG_ENABLED", True):
                raise RuntimeError("Bloomberg disabled in the first cell's settings.")
            size = min(100, max(1, int(env.get("BLOOMBERG_BATCH_SIZE", 100))))
            with env["BloombergClient"]() as client:
                for offset in range(0, len(securities), size):
                    batch = securities[offset:offset + size]
                    for historical, destination in ((False, snapshots), (True, histories)):
                        try:
                            destination.update(_lu_bbg_request(client, batch, start, historical))
                        except Exception as exc:
                            destination.update({s: {"errors": {"request": str(exc)[:600]}} for s in batch})
                try:
                    fx_snapshot = _lu_bbg_request(client, ['USDHKD Curncy'], start,
                                                 reference_fields=['LAST_PRICE', 'LAST_UPDATE_DT'])['USDHKD Curncy']
                except Exception as exc:
                    fx_snapshot = {'errors': {'request': str(exc)[:600]}}
        except Exception as exc:
            fx_snapshot = {'errors': {'request': str(exc)[:600]}}
            for security in securities:
                snapshots.setdefault(security, {"errors": {"request": str(exc)[:600]}})
                histories.setdefault(security, {"errors": {"request": str(exc)[:600]}})
    retrieved = datetime.now(env["HK"]).strftime("%d %b %Y %H:%M:%S HKT")
    fx = _lu_fx_rate(fx_snapshot, start)
    metrics = {s: _lu_performance(snapshots[s], histories[s], start) for s in securities}
    if rows:
        print("Second email: checking A-share capital classes ...", flush=True)
    try:
        a_shares = _lu_a_shares([r['code'] for r in rows], env)
    except Exception as exc:
        # Reference-data failures must not prevent saving the separate email.
        a_shares = {r['code']: {'status': 'N/A', 'error': str(exc)[:600]} for r in rows}
    for row in rows:
        row["price_performance"] = metrics[env["bloomberg_security"](row["code"])]
        row['a_share'] = a_shares[row['code']]
        row['avg_turnover_20d_usd'] = _lu_turnover_usd(row.get('avg_turnover_20d_hkd'), fx)
    subject, body, plain = _lu_render(report, rows, start, end, retrieved)
    to, cc = str(env.get("LOCKUP_EMAIL_TO", "")), str(env.get("LOCKUP_EMAIL_CC", ""))
    message = EmailMessage(policy=SMTP)
    message['Subject'], message['Date'], message['Message-ID'] = subject, format_datetime(now), make_msgid()
    if to.strip():
        message['To'] = to.replace(';', ',')
    if cc.strip():
        message['Cc'] = cc.replace(';', ',')
    message['X-Unsent'] = '1'
    message.set_content(plain)
    message.add_alternative(body, subtype='html')
    folder = Path(first['email_html']).resolve().parent / 'Shortable_Lockups'
    folder.mkdir(parents=True, exist_ok=True)
    # Separate names and folder: never overwrite first-email outputs.
    stem = 'Shortable_Lockups_' + now.strftime('%Y-%m-%d_%H%M%S_%f')
    paths = {suffix: folder / (stem + suffix) for suffix in ('.html', '.eml', '.txt', '.json')}
    audit = {"source_report_date": report['day'], "source_generated_at": report.get('generated_at'),
             "window_start": start.isoformat(), "window_end": end.isoformat(), "months": report['months'],
             "shortability_effective": report['short']['effective'], "bloomberg_retrieved_at": retrieved,
             "source_asof": report.get('source_asof'), "manual_sources": report.get('manual_sources'),
             "cached_sources": report.get('cached_sources'), "original_bloomberg": report.get('bloomberg'),
             "snapshots": snapshots, "histories": histories, "rows": rows,
             "a_share_profiles": a_shares,
             "turnover_fx": fx, "turnover_fx_snapshot": fx_snapshot,
             "turnover_method": "Reuse cell 1 avg_turnover_20d_hkd / current Bloomberg USDHKD LAST_PRICE / 1,000,000. USD millions at the shared snapshot FX rate, not daily historical FX conversion. Original HKD liquidity ratios unchanged.",
             "price_method": "Calendar 7D/1M; last price / observed prior close - 1. Capital changes adjusted; cash dividends excluded.",
             "ipo_price_field": "EQY_INIT_PO_SH_PX",
             "ipo_comparison_method": "(LAST_PRICE / EQY_INIT_PO_SH_PX - 1) * 100; reported offer price, no separate corporate-action adjustment or holder-cost assumption.",
             "max_baseline_age_days": 10, "max_quote_age_days": 10}
    for suffix, content in {'.html': body.encode('utf-8'), '.eml': message.as_bytes(),
                            '.txt': plain.encode('utf-8'), '.json': json.dumps(audit, ensure_ascii=False, indent=2).encode('utf-8')}.items():
        env['atomic_write'](paths[suffix], content)
    opened, detail = (False, "Outlook opening disabled; open the saved .eml to review.")
    if open_outlook:
        opened, detail = _lu_outlook(paths['.html'], subject, to, cc, env)
    print(f"\nSECOND EMAIL READY: {len(rows)} tranches / {len(securities)} stocks")
    print(f"Expiry window: {start} to {end}; YES and NEW included.")
    print(detail)
    print("Email file:", paths['.eml'])
    print("HTML preview:", paths['.html'])
    if any(r['price_performance']['errors'] for r in rows):
        print("Some Bloomberg values are unavailable or need review; see N/A in the draft and details in its JSON file.")
    if any(r['a_share'].get('error') for r in rows):
        print("Some A-share lookups could not be verified. The draft is saved; details are in its JSON file.")
    if rows and fx['errors']:
        print("USD/HKD is unavailable or stale. USD turnover shows N/A; the draft and other values are saved.")
    if not opened and env.get('SHOW_NOTEBOOK_PREVIEW_IF_NO_OUTLOOK', True):
        try:
            from IPython.display import HTML, display
            display(HTML(body))
        except ImportError:
            pass
    return {"email_html": paths['.html'], "eml": paths['.eml'], "audit": paths['.json'], "rows": rows}


# Running this second cell creates ONLY the separate lock-up draft.
if __name__ == '__main__':
    LAST_LOCKUP_DRAFT = _lu_run(globals().get('LAST_DRAFT'), globals(), LOCKUP_EMAIL_OPEN_OUTLOOK)
